# Phase 7 (PCR) - Knowledge distillation: Model 1 + LoRA student, Model 4 (LoRA + LoRMA) teachers
Teacher: the two finished Model 4 runs (Model 4 + LoRA and Model 4 + LoRMA). Their embeddings are computed once, and their cosine scores are averaged into one teacher score. Student: the smaller Phase-1 Model 1 with a fresh LoRA and projection. The student is trained with the same multi-positive contrastive loss as before **plus** a distillation loss: for every query, the student's softmax over the candidates in the step is pulled toward the teacher's softmax (KL divergence).

**Attach as inputs:**
1. The Phase-7 data-prep output (`tokenized_pcr.pkl`)
2. Phase-1 Model 1 folder (`legal-gqa-final`) and its code `legal_gqa_model.py`
3. Phase-1 Model 4 folder (`legal-gqamoe-final`) and its three code files `model.py`, `moe.py`, `legal_gqa_model.py` (in one folder)
4. The outputs of the finished **Model 4 + LoRA** and **Model 4 + LoRMA** runs (each has `adapter_best.pt` and `run_summary.json`)

**Output:** `/kaggle/working/pcr_m1_lora_kd/` (student adapter, dev sweep, `prediction.json`, `submission.zip`, `teacher_embeddings.pt`, `run_summary.json`)

**Why this is a valid distillation setup:** the teachers are the stronger and larger retrieval models (Model 4 has a Mixture-of-Experts feed-forward, see the parameter counts printed below); the student is the smaller Model 1. The student never sees the teachers' weights, only their scores.

**Everything else is as in the Model 1 + LoRA notebook** (document view from the data insights, self-masking, multi-positive loss with false-negative masking, hard + in-batch negatives, dev-selected inference options).

## 1. Configuration

In [ ]:
import os
import sys
import json
import glob
import math
import time
import random
import shutil
import pickle
import zipfile

import numpy as np

# Env vars are only for local testing. On Kaggle leave them alone.
DATA_DIR = os.environ.get("PCR_DATA_DIR", None)
MODEL_DIR = os.environ.get("PCR_MODEL_DIR", None)
MODEL_PY = os.environ.get("PCR_MODEL_PY", None)
OUT_DIR = os.environ.get("PCR_OUT_DIR", "/kaggle/working/pcr_m1_lora_kd")

MODEL_NAME_HINT = "legal-gqa"     # folder name fragment of the Phase-1 Model 1 folder
MODEL_PY_NAME = "legal_gqa_model.py"

SMOKE_TEST = False                # True = tiny run to check the code end to end
if os.environ.get("PCR_SMOKE") == "1":
    SMOKE_TEST = True

SEED = 42
RUN_TRAIN = True
RUN_TEST = True
RESUME = True                     # pick up a checkpoint from an attached earlier run
TRAIN_ON_DEV = False              # True = also train on dev queries (then there is no validation, last epoch is used)

# Document representation
DOC_VIEW = "auto"                 # "auto" (from the data-prep view analysis), "head", "tail", "headtail" or "full"
HEAD_CHUNKS = 1
TAIL_CHUNKS = 1
FULL_MAX_CHUNKS = 8               # only for DOC_VIEW = "full"
CHUNK_LEN = 1024                  # tokens per forward pass, including the closing SEP token
POOLING = "mean"                  # "mean" over the chunk tokens or "last" (the closing SEP token)
USE_PROJ = True                   # trainable d x d projection, starts as the identity
TEMPERATURE = 0.05

# Adapter
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 2e-4                         # adapter learning rate
PROJ_LR = 1e-4                    # projection learning rate

# Training
QUERIES_PER_STEP = 8
POS_PER_QUERY = 3                 # positives sampled per query per step (gold sets are much larger for some queries)
HARD_NEG_PER_QUERY = 3
HARD_NEG_RANGE = (3, 40)          # hard negatives come from this slice of the BM25 non-gold list (skips the very top, which may be unlabeled relevant)
EPOCHS = 5
WEIGHT_DECAY = 0.0
WARMUP_FRAC = 0.05
GRAD_CLIP = 1.0
CHUNK_MICRO_BATCH = 8             # chunks per forward pass during training
EVAL_MICRO_BATCH = 16
USE_CHECKPOINT = True             # gradient checkpointing, needed on a 16 GB T4
SAVE_EVERY_STEPS = 25
LOG_EVERY_STEPS = 5
SESSION_CAP_HOURS = 10.5          # the run saves and stops, and resumes on the next run

# Inference: everything below is chosen on dev
TOP_K = 7
NORM_GRID = [("none", 0), ("csls", 5), ("csls", 10), ("dsm", 0.02), ("dsm", 0.05), ("dsm", 0.1)]
HYBRID_LAMBDAS = [0.0, 0.25, 0.5, 1.0, 2.0]    # weight of token-BM25 (head+tail) added to the neural score
BM25_VIEW_TOKENS = 1280
BM25_K1 = 1.5
BM25_B = 0.75

SMOKE_TRAIN_QUERIES = 24
if SMOKE_TEST:
    CHUNK_LEN = 128
    EPOCHS = 2
    QUERIES_PER_STEP = 4
    CHUNK_MICRO_BATCH = 8
    SAVE_EVERY_STEPS = 3
    LOG_EVERY_STEPS = 2
    print("SMOKE_TEST is on: tiny data, 2 epochs, 128-token chunks, to check the code only.")

CONTENT_LEN = CHUNK_LEN - 1       # one position is kept for the closing SEP

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
print("Config loaded.")

In [ ]:
# Knowledge distillation settings
TEACHER_KEYS = ["m4_lora", "m4_lorma"]   # finished runs used as teachers (their scores are averaged)
KD_WEIGHT = 1.0                          # weight of the distillation loss next to the contrastive loss
KD_TEMPERATURE = 0.05                    # softmax temperature for both teacher and student scores
KD_SPLITS = ["train", "dev"]             # splits the teachers embed (train for the loss, dev for the report)
print("KD settings:", TEACHER_KEYS, KD_WEIGHT, KD_TEMPERATURE)

## 2. Find the inputs

In [ ]:
def find_one_file(roots, name):
    hits = []
    for root in roots:
        for dir_path, dir_names, file_names in os.walk(root):
            if name in file_names:
                hits.append(os.path.join(dir_path, name))
    if len(hits) == 0:
        return None
    hits.sort(key=len)
    return hits[0]


INPUT_ROOTS = os.environ.get("PCR_INPUT_ROOTS", "/kaggle/input").split(":")


def find_own_file(name):
    # A file from an earlier run of THIS notebook (same output folder name). Never a file of the teacher runs,
    # which also contain ckpt_pcr.pt / adapter_best.pt with different shapes.
    own = os.path.basename(OUT_DIR.rstrip("/"))
    hits = []
    for root in INPUT_ROOTS:
        for dir_path, dir_names, file_names in os.walk(root):
            if name in file_names and os.path.basename(dir_path) == own:
                hits.append(os.path.join(dir_path, name))
    if len(hits) == 0:
        return None
    hits.sort()
    return hits[0]


def find_m1_py():
    # Model 1 and Model 4 both have a legal_gqa_model.py. The Model 4 one sits next to moe.py, so skip those.
    hits = []
    for root in INPUT_ROOTS + ["/kaggle/working"]:
        for dir_path, dir_names, file_names in os.walk(root):
            if "legal_gqa_model.py" in file_names and "moe.py" not in file_names:
                hits.append(os.path.join(dir_path, "legal_gqa_model.py"))
    if len(hits) == 0:
        return None
    hits.sort(key=len)
    return hits[0]


def find_model_dir(name_hint):
    found = []
    for root in INPUT_ROOTS:
        for dir_path, dir_names, file_names in os.walk(root):
            low = dir_path.lower()
            if "config.json" not in file_names or "checkpoint" in low:
                continue
            model_type = ""
            try:
                with open(os.path.join(dir_path, "config.json")) as f:
                    model_type = str(json.load(f).get("model_type", ""))
            except Exception:
                model_type = ""
            if "moe" in model_type or "moe" in low:
                continue
            if name_hint in low:
                found.append(dir_path)
    if len(found) == 0:
        return None
    found.sort()
    for path in found:
        if "final" in path.lower():
            return path
    return found[0]


if DATA_DIR is None:
    BUNDLE_PATH = find_one_file(INPUT_ROOTS, "tokenized_pcr.pkl")
else:
    BUNDLE_PATH = os.path.join(DATA_DIR, "tokenized_pcr.pkl")
if BUNDLE_PATH is None or not os.path.exists(BUNDLE_PATH):
    raise FileNotFoundError("tokenized_pcr.pkl not found. Attach the Phase-7 data-prep output (Add Input).")
print("data bundle:", BUNDLE_PATH)

if MODEL_DIR is None:
    MODEL_DIR = find_model_dir(MODEL_NAME_HINT)
    if MODEL_DIR is None:
        raise FileNotFoundError("No Phase-1 Model 1 folder with config.json found. Attach legal-gqa-final.")
print("model folder:", MODEL_DIR)

if MODEL_PY is None:
    MODEL_PY = find_m1_py()
    if MODEL_PY is None:
        raise FileNotFoundError("legal_gqa_model.py not found. Attach it (see the header).")
print("model code:", MODEL_PY)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16
USE_SCALER = USE_AMP
torch.manual_seed(SEED)
print("device:", DEVICE, "| fp16 autocast:", USE_AMP)

START_TIME = time.time()
SESSION_CAP_SEC = SESSION_CAP_HOURS * 3600.0
if os.environ.get("PCR_SESSION_CAP_SEC") is not None:
    SESSION_CAP_SEC = float(os.environ.get("PCR_SESSION_CAP_SEC"))

## 3. Load the data bundle and read the data insights

In [ ]:
with open(BUNDLE_PATH, "rb") as f:
    B = pickle.load(f)

SEP_ID = int(B["sep_id"])
PAD_ID = int(B["pad_id"])
VOCAB_SIZE = int(B["vocab_size"])
print("splits:", B["splits"], "| vocab:", VOCAB_SIZE, "| SEP id:", SEP_ID, "| PAD id:", PAD_ID)
if PAD_ID == SEP_ID:
    print("PAD and SEP share an id: padding is always masked by length, never by token id.")

GOLD = B["gold"]
GOLDSET = {}
for split in B["splits"]:
    if split == "test":
        continue
    for qid in GOLD[split]:
        GOLDSET[(split, qid)] = set(GOLD[split][qid])

# What the data says (printed so the choices below can be checked)
for split in ["train", "dev"]:
    if split not in B["splits"]:
        continue
    counts = []
    gold_union = set()
    for qid in GOLD[split]:
        counts.append(len(GOLD[split][qid]))
        for c in GOLD[split][qid]:
            gold_union.add(c)
    arr = np.array(counts)
    pool = len(B["pool_ids"][split])
    print("%s: %d queries | pool %d | gold per query mean %.1f median %.0f max %d | %.0f%% of the pool is gold for some query" % (
        split, len(arr), pool, arr.mean(), np.median(arr), arr.max(), 100.0 * len(gold_union) / pool))

view_results = B.get("view_results", {})
print("data-prep view analysis (BM25 Micro-F1@7 on dev):", view_results)
if DOC_VIEW == "auto":
    best_name = None
    best_score = -1.0
    for name in ["head", "tail", "headtail", "full"]:
        if name in view_results and view_results[name] > best_score:
            best_score = view_results[name]
            best_name = name
    if best_name is None:
        best_name = "headtail"
    DOC_VIEW = best_name
print("document view used:", DOC_VIEW, "| head chunks", HEAD_CHUNKS, "| tail chunks", TAIL_CHUNKS, "| chunk length", CHUNK_LEN)

## 4. Load the frozen Phase-1 Model 1

In [ ]:
import importlib.util
import inspect

sys.path.insert(0, os.path.dirname(MODEL_PY))
spec = importlib.util.spec_from_file_location("legal_gqa_model", MODEL_PY)
legal_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(legal_mod)

LM, load_info = legal_mod.LegalGQAForCausalLM.from_pretrained(MODEL_DIR, output_loading_info=True)
missing_keys = []
for key in load_info.get("missing_keys", []):
    if key != "lm_head.weight":          # tied to the token embedding
        missing_keys.append(key)
print("Phase-1 weights loaded | missing keys:", len(missing_keys), "| unexpected keys:", len(list(load_info.get("unexpected_keys", []))))
if len(missing_keys) > 0:
    raise RuntimeError("Phase-1 weights were NOT fully loaded, missing: " + str(missing_keys[:5]))
if LM.config.vocab_size != VOCAB_SIZE:
    print("WARNING: model vocab", LM.config.vocab_size, "!= data vocab", VOCAB_SIZE)
if LM.config.vocab_size <= SEP_ID:
    raise RuntimeError("data / model vocab mismatch")

# lm_head is tied to the token embedding. If it was not in the checkpoint it must be tied explicitly.
# (lm_head is not used for retrieval, but the tie is kept so the model equals the Phase-1 model.)
if "lm_head.weight" in list(load_info.get("missing_keys", [])):
    LM.lm_head.weight = LM.wte.weight
    print("lm_head.weight was not in the checkpoint -> tied to the token embedding")

LM.float()
LM = LM.to(DEVICE)
LM.eval()
for p in LM.parameters():
    p.requires_grad = False

N_BASE = sum(p.numel() for p in LM.parameters())
D_MODEL = int(LM.wte.weight.shape[1])
print("model:", type(LM).__name__, "| params: %.2fM | hidden size %d" % (N_BASE / 1e6, D_MODEL))

for attr in ["max_position_embeddings", "n_positions", "block_size", "max_seq_len", "context_length", "n_ctx"]:
    value = getattr(LM.config, attr, None)
    if isinstance(value, int) and value > 0:
        print("model context (%s): %d" % (attr, value))
        if CHUNK_LEN > value:
            raise RuntimeError("CHUNK_LEN %d is longer than the model context %d" % (CHUNK_LEN, value))
        break


def backbone_hidden(input_ids):
    # Same computation as the Phase-1 forward(), stopped before lm_head. Causal model, right padding is harmless.
    T = input_ids.shape[1]
    x = LM.drop(LM.wte(input_ids))
    cos, sin = LM._rope_slice(0, T, input_ids.device, x.dtype)
    for blk in LM.blocks:
        if torch.is_grad_enabled() and USE_CHECKPOINT:
            def run(h, blk=blk):
                return blk(h, cos, sin, None, True, None, False)[0]
            x = torch_checkpoint(run, x, use_reentrant=False)
        else:
            x = blk(x, cos, sin, None, True, None, False)[0]
    return LM.ln_f(x)


# My manual forward must agree with the model's own forward.
with torch.no_grad():
    probe = torch.tensor(np.random.randint(3, LM.config.vocab_size, size=(1, 48)), device=DEVICE)
    own_logits = LM.lm_head(backbone_hidden(probe)).float()
    ref_logits = LM(input_ids=probe).logits.float()
    agree = float((own_logits.argmax(-1) == ref_logits.argmax(-1)).float().mean())
    diff = float((own_logits - ref_logits).abs().max())
print("manual forward vs model forward: top-1 agreement %.3f | max logit diff %.4f" % (agree, diff))
assert agree > 0.9, "manual forward does not match the model forward"

## 5. Document views and the encoder
A case is cut into chunks of `CHUNK_LEN` tokens. Each chunk is run through the model, pooled, projected and normalised; a case embedding is the mean of its chunk embeddings.

In [ ]:
def chunk_starts(L, mode):
    CL = CONTENT_LEN
    if L <= CL:
        return [0]
    n_cover = int(math.ceil(L / CL))
    starts = []
    if mode == "full":
        if n_cover <= FULL_MAX_CHUNKS:
            for i in range(n_cover):
                starts.append(min(i * CL, L - CL))
        else:
            for s in np.linspace(0, L - CL, FULL_MAX_CHUNKS):
                starts.append(int(round(s)))
    else:
        n_head = 0
        n_tail = 0
        if mode in ["head", "headtail"]:
            n_head = HEAD_CHUNKS
        if mode in ["tail", "headtail"]:
            n_tail = TAIL_CHUNKS
        if n_cover <= n_head + n_tail:
            for i in range(n_cover):
                starts.append(min(i * CL, L - CL))
        else:
            for i in range(n_head):
                starts.append(i * CL)
            for j in range(n_tail):
                starts.append(max(L - (j + 1) * CL, 0))
    starts = sorted(set(starts))
    return starts


CHUNK_CACHE = {}


def get_chunks(spec):
    # spec = (split, "q" or "c", id)
    if spec in CHUNK_CACHE:
        return CHUNK_CACHE[spec]
    split, kind, doc_id = spec
    if kind == "q":
        ids = B["tokens"][split]["queries"][doc_id]
    else:
        ids = B["tokens"][split]["cands"][doc_id]
    chunks = []
    for s in chunk_starts(len(ids), DOC_VIEW):
        chunks.append(ids[s:s + CONTENT_LEN])
    CHUNK_CACHE[spec] = chunks
    return chunks


def make_batch(group):
    longest = 0
    for c in group:
        if len(c) > longest:
            longest = len(c)
    T = longest + 1
    ids = np.full((len(group), T), PAD_ID, dtype=np.int64)
    lengths = np.zeros(len(group), dtype=np.int64)
    for i, c in enumerate(group):
        L = len(c)
        ids[i, :L] = c.astype(np.int64)
        ids[i, L] = SEP_ID
        lengths[i] = L + 1
    return torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lengths).to(DEVICE)


def pool_hidden(h, lengths):
    T = h.shape[1]
    if POOLING == "last":
        rows = torch.arange(h.shape[0], device=h.device)
        return h[rows, lengths - 1]
    positions = torch.arange(T, device=h.device).unsqueeze(0)
    mask = (positions < lengths.unsqueeze(1)).to(h.dtype)
    summed = (h * mask.unsqueeze(-1)).sum(1)
    return summed / lengths.unsqueeze(-1).to(h.dtype)


class Projection(nn.Module):
    # d x d linear map that starts as the identity, so training starts at the plain Phase-1 pooled representation
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(d, d))
        with torch.no_grad():
            for i in range(d):
                self.weight[i, i] = 1.0

    def forward(self, x):
        return x @ self.weight.t()


PROJ_MODULE = None
if USE_PROJ:
    PROJ_MODULE = Projection(D_MODEL).to(DEVICE)


def pooled_chunks(chunks, grad):
    n = len(chunks)
    order = list(range(n))
    if not grad:
        order.sort(key=lambda i: len(chunks[i]))      # similar lengths together, less padding
    micro = CHUNK_MICRO_BATCH if grad else EVAL_MICRO_BATCH
    pieces = []
    piece_index = []
    for start in range(0, n, micro):
        idx = order[start:start + micro]
        group = []
        for i in idx:
            group.append(chunks[i])
        ids, lengths = make_batch(group)
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            hidden = backbone_hidden(ids)
        pooled = pool_hidden(hidden.float(), lengths)
        if PROJ_MODULE is not None:
            pooled = PROJ_MODULE(pooled)
        pooled = F.normalize(pooled, dim=-1)
        pieces.append(pooled)
        for i in idx:
            piece_index.append(i)
    all_pooled = torch.cat(pieces, 0)
    inverse = torch.empty(n, dtype=torch.long)
    inverse[torch.tensor(piece_index, dtype=torch.long)] = torch.arange(n)
    return all_pooled[inverse.to(all_pooled.device)]


def embed_docs(specs, grad):
    chunks = []
    owner = []
    for k, spec in enumerate(specs):
        for c in get_chunks(spec):
            chunks.append(c)
            owner.append(k)
    chunk_emb = pooled_chunks(chunks, grad)
    owner_t = torch.tensor(owner, dtype=torch.long, device=chunk_emb.device)
    zeros = torch.zeros(len(specs), chunk_emb.shape[1], device=chunk_emb.device)
    summed = zeros.index_add(0, owner_t, chunk_emb)
    return F.normalize(summed, dim=-1)


def embed_split(split, max_queries=None):
    qids = list(B["query_ids"][split])
    if max_queries is not None:
        qids = qids[:max_queries]
    pids = list(B["pool_ids"][split])
    q_specs = []
    for q in qids:
        q_specs.append((split, "q", q))
    p_specs = []
    for p in pids:
        p_specs.append((split, "c", p))
    with torch.no_grad():
        Qe = embed_docs(q_specs, False).cpu()
        Pe = embed_docs(p_specs, False).cpu()
    return qids, pids, Qe, Pe


# how many chunks does the chosen view give per document?
for split in B["splits"]:
    n_q = []
    for q in B["query_ids"][split][:200]:
        n_q.append(len(get_chunks((split, "q", q))))
    n_c = []
    for p in B["pool_ids"][split][:200]:
        n_c.append(len(get_chunks((split, "c", p))))
    print("%s: chunks per query %.2f | per candidate %.2f (view %s)" % (split, np.mean(n_q), np.mean(n_c), DOC_VIEW))

## 6. Scoring, ranking and the dev-selected inference options
Plain cosine is always available. Optional steps (each chosen on dev): centring, CSLS / dual-softmax normalisation across queries, and a token-level BM25 (head+tail) hybrid.

In [ ]:
NEG_INF = float("-inf")


def make_scores(Qe, Pe, qids, pids, center):
    Q = Qe
    P = Pe
    if center:
        mu = torch.cat([Q, P], 0).mean(0, keepdim=True)
        Q = F.normalize(Q - mu, dim=-1)
        P = F.normalize(P - mu, dim=-1)
    S = Q @ P.t()
    pos_of = {}
    for i, p in enumerate(pids):
        pos_of[p] = i
    for r, q in enumerate(qids):
        if q in pos_of:
            S[r, pos_of[q]] = NEG_INF        # a query is in its own pool: never retrieve itself
    return S


def apply_norm(S, norm, param):
    if norm == "none":
        return S
    finite = torch.isfinite(S)
    if norm == "csls":
        low = S.masked_fill(~finite, -1.0)
        k = min(int(param), S.shape[0])
        r_cand = low.topk(k, dim=0).values.mean(0, keepdim=True)
        out = 2.0 * S - r_cand
        return out.masked_fill(~finite, NEG_INF)
    if norm == "dsm":
        z = S / float(param)
        out = torch.log_softmax(z, dim=1) + torch.log_softmax(z, dim=0)
        return out.masked_fill(~finite, NEG_INF)
    raise ValueError(norm)


def zscore_rows(S):
    finite = torch.isfinite(S)
    count = finite.sum(1, keepdim=True).clamp(min=1).to(S.dtype)
    filled = S.masked_fill(~finite, 0.0)
    mean = filled.sum(1, keepdim=True) / count
    centered = (filled - mean) * finite.to(S.dtype)
    var = (centered ** 2).sum(1, keepdim=True) / count
    out = (S - mean) / (var.sqrt() + 1e-6)
    return out.masked_fill(~finite, NEG_INF)


def top_ids(S, pids, k):
    k = min(k, S.shape[1])
    idx = S.topk(k, dim=1).indices
    out = []
    for r in range(idx.shape[0]):
        row = []
        for j in idx[r].tolist():
            row.append(pids[j])
        out.append(row)
    return out


def micro_prf(pred_lists, qids, gold, k):
    tp = 0
    n_gold = 0
    n_pred = 0
    for r, q in enumerate(qids):
        g = set(gold[q])
        n_gold += len(g)
        n_pred += k
        for c in pred_lists[r][:k]:
            if c in g:
                tp += 1
    p = tp / max(n_pred, 1)
    rec = tp / max(n_gold, 1)
    f1 = 0.0
    if p + rec > 0:
        f1 = 2 * p * rec / (p + rec)
    return p, rec, f1


def recall_at(S, qids, pids, gold, k):
    lists = top_ids(S, pids, k)
    tp = 0
    n_gold = 0
    for r, q in enumerate(qids):
        g = set(gold[q])
        n_gold += len(g)
        for c in lists[r]:
            if c in g:
                tp += 1
    return tp / max(n_gold, 1)


# ---- token-level BM25 over the head+tail token view (no tokenizer needed, terms are token ids)
import scipy.sparse as sp


def bm25_view(ids):
    half = BM25_VIEW_TOKENS // 2
    if len(ids) <= BM25_VIEW_TOKENS:
        return ids
    return np.concatenate([ids[:half], ids[-half:]])


def count_matrix(arrays):
    indptr = [0]
    indices = []
    data = []
    for ids in arrays:
        view = bm25_view(ids)
        view = view[view != SEP_ID]
        if len(view) == 0:
            indptr.append(indptr[-1])
            continue
        terms, counts = np.unique(view, return_counts=True)
        indices.append(terms.astype(np.int64))
        data.append(counts.astype(np.float32))
        indptr.append(indptr[-1] + len(terms))
    if len(indices) == 0:
        indices_all = np.zeros(0, dtype=np.int64)
        data_all = np.zeros(0, dtype=np.float32)
    else:
        indices_all = np.concatenate(indices)
        data_all = np.concatenate(data)
    return sp.csr_matrix((data_all, indices_all, np.array(indptr)), shape=(len(arrays), VOCAB_SIZE))


def bm25_scores(split, qids, pids):
    P_arrays = []
    for p in pids:
        P_arrays.append(B["tokens"][split]["cands"][p])
    Q_arrays = []
    for q in qids:
        Q_arrays.append(B["tokens"][split]["queries"][q])
    X = count_matrix(P_arrays).tocsr()
    n_docs = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float64)
    idf = np.log(1.0 + (n_docs - df + 0.5) / (df + 0.5))
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
    avgdl = max(dl.mean(), 1.0)
    row_of_nz = np.repeat(np.arange(n_docs), np.diff(X.indptr))
    tf = X.data.astype(np.float64)
    denom = tf + BM25_K1 * (1.0 - BM25_B + BM25_B * dl[row_of_nz] / avgdl)
    weights = idf[X.indices] * tf * (BM25_K1 + 1.0) / denom
    W = sp.csr_matrix((weights.astype(np.float32), X.indices.copy(), X.indptr.copy()), shape=X.shape)
    Q = count_matrix(Q_arrays).tocsr()
    Q.data = np.log1p(Q.data)
    S = torch.from_numpy((Q @ W.T).toarray().astype(np.float32))
    pos_of = {}
    for i, p in enumerate(pids):
        pos_of[p] = i
    for r, q in enumerate(qids):
        if q in pos_of:
            S[r, pos_of[q]] = NEG_INF
    return S


def run_sweep(Qe, Pe, qids, pids, gold, bm25_S):
    bm25_Z = zscore_rows(bm25_S)
    rows = []
    order = 0
    for center in [False, True]:
        S0 = make_scores(Qe, Pe, qids, pids, center)
        for norm, param in NORM_GRID:
            S1 = zscore_rows(apply_norm(S0, norm, param))
            for lam in HYBRID_LAMBDAS:
                if lam == 0.0:
                    S2 = S1
                else:
                    S2 = S1 + lam * bm25_Z
                    S2 = S2.masked_fill(~torch.isfinite(S1), NEG_INF)
                lists = top_ids(S2, pids, TOP_K)
                p, r, f1 = micro_prf(lists, qids, gold, TOP_K)
                rows.append({"order": order, "center": center, "norm": norm, "param": param, "lambda": lam,
                             "precision": p, "recall": r, "f1": f1})
                order += 1
    rows.sort(key=lambda d: (-d["f1"], d["order"]))      # ties keep the simpler option first
    return rows


def final_scores(Qe, Pe, qids, pids, cfg, bm25_S):
    S0 = make_scores(Qe, Pe, qids, pids, cfg["center"])
    S1 = zscore_rows(apply_norm(S0, cfg["norm"], cfg["param"]))
    if cfg["lambda"] == 0.0:
        return S1
    S2 = S1 + cfg["lambda"] * zscore_rows(bm25_S)
    return S2.masked_fill(~torch.isfinite(S1), NEG_INF)


def plain_f1(Qe, Pe, qids, pids, gold):
    S = make_scores(Qe, Pe, qids, pids, False)
    lists = top_ids(S, pids, TOP_K)
    return micro_prf(lists, qids, gold, TOP_K)

## 6b. Teachers
The two Model 4 runs are rebuilt from their saved adapters and used to embed every train and dev case once. The teacher score of a query/candidate pair is the average of the two teachers' cosine similarities.

In [ ]:
# Code of the finished runs (config, model loading, encoder, scoring, adapter). Replayed in separate namespaces.
_SOURCES_B64 = {}
_SOURCES_B64['m4_lora'] = (
    'eNrVfQt320ay5l/BKMdnAZuiHrGTGd4oO7RM29roZUnO41I8OBAJSoj4gAnSspLJ/vb7VVW/AUry7N6zZz0TmwC6q7urq6urqquq'
    '+xvFtJwvltG8upypn9W9/f17NZ+Zh+vJ/Mo8TLPljXlYFtPcPCyy2Wg+teBuVstiYh7LYng7sYX/KMpxQc/mzWw1Le+jrIpmJb39'
    'JurNPkefs0UVZYs8ms8m99F4vogm82E2iZZ5tSxm1+3oZBb9lF1fT/Jokmef82h5k0+jbDKf5e3L2ZvuRTd9c3AW7WGc7Xz2uVjM'
    'Z+3rfBlfbpzun6X6++VGKzpGleRydnTypnf4UB1ToFbp9LeH65z+5lQ5+XjxUCvqM1W43Ni65RFu3c0Xtxj0VjlcpNOX6WS+yC43'
    'EkKWNHDcPeql7w+OLwD1cmOSX2eTzetP2XSeX25E0TdA32SUL6JZNs2j8SK7nuYzEMCYcBad3mRVvrkTHc1H+SR6qcpqyBe/nfYs'
    '0BRA0+kN/tOQp1QrXd6XeXS3KJbLfBYV+P+yiobz2bi4bgs9Xc7Oj05+6qUXvXPq49tsUuVR8Oeb6GKxyvEV83sfLVazaDmPhjf5'
    '8Jb7OURLUT4b0Vv8A+oZN2OQWwJ6oj3q987lRudyRvC9HlBT3K1e7w0eX+5ezs4+HqcXZ92DY/OZ37jlz3rnH4966jFq+vMNk3u0'
    'KqNM+l7OC+B6vJiDOGdRtlxmeD2K8mwxKTAlGObljFtNT47TN72fm7FjcINP82i5yIDk+Swa5Z+jT6t8UeRVFANJM8IU1kyBtTTH'
    'EpoUo2xZzGetaJJVyygv58Mb+riq8lEia+3NfLhicljk5SKv8ItrYAmd7Kc/H/R+4dnPVss5TXhttOZTzEOkiUKT2SZgldHnIr/D'
    'qLPJfVVUCRH0TZ6NhLSXGVjEhn4nTxFW+eXGeDXB78vZ+173Tbr//uPxT+foww7Q1D049F68/Xh4mB51f7Uv/17roGEf3nh0G1wx'
    'PezRnO9s775smtDl/DafVVGJ2QKcu2wxisqsqlog9OFkNcK6FPqczCv6fd47lSqXs9OTk8OD43fc4jTPZg0YZBSqb/PPaIJB3axm'
    't7pdxglNHyG5sSHM5MfzXnp6dvK/1pPmN0I12RUY5ij6gv/Kxfz3fCjkUS2zBdYseDC1UIxABsXyHijvHZ32zroXH8+I6rfb26+E'
    'arqjrFwSlzg8OeumxMx2vlMP3cPT9128+HZXvXhzdnIKrmbqH1Lx3XzzZbTuzzdRJvCJtS9mNNhFtsT6oyGmXH9nTf1vnGGFtanj'
    'F4QEvLqcffjYOzvonacYX3p+AVyCemjK5A19Ja7+bVMLwP+y+Iw1V2XTcoLVTLRB6/Cef1VL0H58DT4aVflSdrHpCitvki2uhYqi'
    'ag5WrJYu5u999+xNetx757ftvD/rHr+jKYi/bUUvtxO3OzdEkTOwZ+nTcM5Mnhcjlno1KYa55vWvj3ZfgTPMNrlzkwI8Ia5ui1Km'
    '/TMNYDkvW9HdTYHuTrP76CqPVrNJdpXTMBf4+3M2W6LDvdOT/fe04jChv/QO3r3HrtXb7/4ms4x33bOjj6fp27Puvpn4d2e0oA8P'
    'CNU7VEhW39HB/tlJ+rp7sf++toK/kbVQX33RaLXglWfms/dz9zCARTRJK2P/fW//p9MT2R9r6+Ob6HqRjQrigZZjA2QLSM1HGDYo'
    'KQOs6N3r6OIltozuz7209zPmiOmGkLBLZH3yLnj7iraX8/MDMPb97mn6/uTjGTOu7fYrb10C9bTbVRlNH0QpENC8BH+hn2DJYM8V'
    '9YHKzfIvS9kziJgPZmNw+9kw70Q5TR4mHCjBXM3viMsPb+bg52qfwFo+OU1/QvvfX86OT86O0ndnB7T19bFtgiRyYsbbYNJ4HFaT'
    'ih5f+Y876vOomnLh9vZu+OJV8GInGYCIf3uNptLD7tHrN13CQB8l6evuK/obf+3Q8257eyAYucuL6xsRUIi/bTLZxrRPvKCNIgF7'
    'oHmBICA4WS0gGFbD+QKLnMoyn08vsN0fM8J3/76tPvy0w7T3Sj2+Ztr8/pUjoPBWrHgDzexLFjSs7KCECW/j2P27vDSLYleeG1jM'
    'S7d6QPlKTKmT17fypU5hqp0SS4GkH0fCKYhkOiJK0YYM9Mr2D7pCfzcZsWpxtRoELdo220rG3D85vugdX6jR2pFvRjvONpsrvghy'
    'Q+O3eblkRhfsWAQPUts0u81HxaKKlbjbivIv4Ebp/HaP1ieaFZ2iXWEJxiSk4dWsbDe81YPfZ3kTWkIG2qCugwA3Rvk4GhezUYr+'
    'paR2xIv5fIkRkyicqLm8IXEVVDmQR+o1lSJBlkurYvob+p2W0IVa/IsAAR7Blt9UDSO8yya33FjiVKc/oCaWw1HMVgrK6F61s7KE'
    'rBsDHrXY/h2sKbbt8yASqQqwk3wWUy2WfLcdkIt8uVrMWAGxI25X0L3i2/x+b8IihFOQPve3B5fMZgwKRdQv72M9IiX9t6G8YcYz'
    '2T+w/UxBAtQ1iBVzkb+y4ZDoIdOKyPImW4ooC+UDUHKCQezO0TIU6Pqc9K1eVMzK1bJRVbrcGPxfnzYIarpTG/70cd/pc77+Y8PQ'
    'Nh4lAjUha+bf7ZA/gzLVDdOHyjE1lmKrMGMcz1foor8CvgJdtflwcUd70Z6B1sZjvogT+53x6uqKGxBQlgEGWQa2m7MgDqACjAEM'
    'GN4qt28d9ZSEcBL59afl4j6oflcsb6I5Fly8Ft9eR5OEBOZxw7R5zVLhNrGleJwoNdV+F+I100d/8i+8WHr8D9hpAP+BIQGX7te9'
    'yNHh56L+87w3Y4+pQHMcPWifu3CRx9kL0Q9V5lmkKp1gvkGQ2URm0aWKoEMKbsl2J+cFgzT8CQCNtQdsiHqh4Lz+ePzmEHpR94K2'
    'V38bqDORAWultC0Wf+SjFKaWdnk74ZnJoY83wvToRPdiPRx01a2vestTM18aYLwTVrFTUiMGsi4MA2/R/+P58i1hobdYzBdxQ3sM'
    'UWY06rLRwTH2fO/o6PPVEsOP4u5oBHkSPxPZ9vW2SiWjK8CZAAUgVrdbCv3Wcubj37WoBSwoMFtZKlsH6+HhH8/XmLGi2FkQNRNW'
    'EmDINZxtMo36uGBYCjIjw/TWRwXsgU2YYDNhsJUGA6/VfHjcerAsssVmN36ht9QXDRsqc25QnQwjaSCUgrTCnC2qEQndKNWEB2rU'
    'wcLpb4lj013OF8Mb/6k9m7GRd1Z73R6vZqyxQ4xHibeXM9Ff+TPZkqu2Y0tTlZ03ZLagsqmzRZAFq/fzwT4pzQIIShBUYdJnVqOM'
    'mM9YfaDndlGl2WcoF2QgiZOIljzx+3LFIyctsntEArwAbWsOq6FdzvA5faPMpQJ3DKa/1Dro+X73sEcrQYG6nEmhaTZbYYaaJFrp'
    'MKNYWiXO8q9oXEIXJavbEEYh/qpAMv7PL7pnF+nFAZspyUrfpr+I0Fxd9LxHanldO30effvdNivwa62rPhSys7K1celSbb0pxkX8'
    'RIgsutvd2GE5hIHFFTXq7Lykxckhg95mxbJ7mrKCS8h8DX5f5WVawPo4wOdTWCK8j2U2Mh9/Ptnvvk7PD/6z5xT4DHRfpRV4rBTS'
    'k1SVk4JUBGKNff2odpN/RVyLP1qg8oWsd8WIP0lP5TU6pl9LH2Xb0P3dU4U7vuJHtUjKJKDVDdmbYGsGGOydIzZQknw+ucvuIWxn'
    'FVSv6OqeNvRr1iLIdkAvRCcsRlrve3dySCiiYZG9CIOSd+c9sqP8+ReWKVmyaMTEVLzhdwxjk++8VOjsxpjj6wIbQftUjAgWNdPn'
    'mq4Yrxrvx/ylRYWTAboCK1vs1Ojj/UCZt38hTUNbpWFiAQZixhpwUM2VtXWOZVYpu8kQmLvKhbuwkdwbI4bIFicR28isUh+qEl+b'
    '0FEf8xCc19M+CdPpakZqtAwseSJyBJKW30haq6Ek8TWiYQCMCwUymO1OG6QUDxWIbLFA96CQ40d2H0vb6ls5n0/wkXrAKwuPWFqE'
    'BdVO4hPvs6oTPRuZc4x/CQC8+Re37hhXyU4ePWvvQMyF9YB/b+N39kVK09OzZ9rWyVBA9wzDM7mS4vUMZiozUEVO1GOMByYsqtui'
    'QbapyRhvMFRpU5Xgb9kX+rRDDBOck6pbdCXRFoNJmBDpJCQlW96EJ/u1Yn/uayKpP/8KJK+Gc5QoZnPYUTFczDff7vzze2XgS5hp'
    'uBCFddhzD3uQoyb5CuuRNSx0yUrv/JataXi9yYZaTTHacNF/7CSnZc5YBr7sryF4CCHW5b7oU6lB9KPTlYAsvT7WqzYUVsOkf8x6'
    'tR/qQpdbyR2ZlHAOk0xBZ+b0mRpPHB22ySauKgmnJ5DKAEcfndMu+U7NOd+dwy/5LodEwsSpgLHL8e6phCT5Z1Jctb0j+WJWlTgj'
    'IcrE2b+oHXiXwyAEQ6xWRCCo07hiI9+BoKge7Yoe3Da9TUlmY+UqpZN6EudibfWAOCryZyAu6q+jOki8XAEUAyXwMf2VSH/lW9Vv'
    'AD9gzqOgSndZKMgX0KvQSakamyK8Og+PWmw3TAtwexdA+5B+vfvQPZr33s4X+9mqyiaHR23uFZYm7wUQ3Iwe0FL6VErgsPEyRG3S'
    'nBYVmUFT2Nw0x6dFhUdRyFUHtI3AKU1o6w8Su9NQnb/xsfw0JUJqi8kcK9s7WSislVw293x6lbNE4JhInHb07oHfDiPSqpW0USkT'
    'Kxiuqkq9EQmIWKALLxFSXeHYgugNlbyidAAVh+O2hd2hJ4lws1oL4BLbnnp8hq0IEq9WkcLO39H5+PHJRUTs6V6NpaWH0kF/oTVV'
    'y4XXSr/zaqA6gNlXRiArEdJUWPkukM1wEnaMQ+COmGZEIqRBNQEidAEWyyqmoIX8QA9+CCTDRlww3C23IzRu+PVAVdtQApMiKOKI'
    'D5BPOzoYk554l1Va3hFRymppcBha4XwRshTDwaTiJLJY0rECmomddggAMUmx8mJRFThlxP57tRLRDfXNkYKS2WQI+SeoTpXnxSJa'
    'riCqvjp4nTURXcNi06sN6PbBYOni3d0y12C96Q7KrkfQ5o8Prk89Hxf4pmw0k+weohBjVYT8gulZ0TWpSw2AwJHIJ+SOyEHj/G6+'
    'gkyEmVHHKZqs/L63iVzSEkshIQL3xux8U2jCWNLrrDSqXlzH26YPI2lnV1WciBhlDSEa0I++jbGJngMaegCdOI4HyGftl+PkPzx6'
    '8dgCAYEcAcU8H7GMqPri8ELd5ANtdaL5rZo+2ioYGwn9FsJZzmPR5vldm0g9VmpGSWSCd2W2wL4L/4fKoBdHX6D2Ar1L6bRaewlR'
    'I8c4QTwnVRUnxXHZhjddDojROoBo6434pSnt1p9YUFaZ93cGobmHmTZZPVA+aacs8aSp8HcGT1J8e3d8hH3hpsD5LA60iTE9E0zG'
    'qpdb8Nz4DqKQ9ECEY+opHKMWSrIEPaT6KDE1WK1EqJyZT+rFFYSNW6WX84kIalf5pxQbhbwgfQun5qmVlAjKcPnFyqaYAnauAi+g'
    'fsSGw7a4X8Z1T1FoAVYBhxmcu8dckzyBlolIsQzJJ93QasbdieJnVdIx2JFmuHbiH47Y49Yf5XtoJG9aGLbSM14d8Iq8lmO3mcM+'
    'dVd0J0ythp5cLfLs1p4oXWXD2ysypstUx2xDJy3Png2ek+yM00B8YGFQOxfplae8OUiDIn+HkowS+Zgkes02IpG3pLetaMFMxLFo'
    'gAdOIQZWbWnygulZ9cPQsXz7ImtvtJiXit6dLidaFccJF6ZWiqYomqfsPUMy8UXLgS2GuVb0pT2iFeFo6FeTW7XkmC4rX/kRi18h'
    'SzjN2RkLCGDK8f1UgkkmlMPrI4a1Bi3s4b9k/WEhvlJBPRyh3hb7vejfzDsSPkJxAXzRpkvHkBqjXYy0RTs0lCxoNdg0lnsCwTm0'
    'skckLjTqy5ev6osaBO0es3Qcf0kulbPLEfR/tpUaPyDeCbNrdMrufkwr/wMnhnczXQ7kIZ/F2jxn7FvGuphf5cZkC/9VHIvH1tOA'
    '/iEsfLtOYKO/9+IdeGX9PQEpC2nsGQ5PTaAv0AeuxcvAbotxuIi4K0litwzBx9itbMl2T4q35aNfSZBitmLbAdhrrmm/3dzhUzwL'
    '3PlgOqCMHwrmqBiPm0BiW7dgwj3d8D5/5j5XkT5L4RcdcjsDT+B+s+b8rP3tmDQM2HYYsjRPG7himFSyxW+pHXiEQXlV4wb7bf9D'
    '9gKv1dE8F0GBJV6HEaoClxvaZYR161ScIuPDFhfTBLNP+6bjEmO2hUMSwvcP6wejhrax7bCPp+y75E/fHubFJD7Etrh/qBGtfDGt'
    'SVAd64oBR6wqgVFFwUX7gUts7YgXmyz7tWTYEGJVr4mXSCe0PjjF8WYBE9c+cHGICbedXbP22d5GDWElTcjakAkblcqtsJuP94A5'
    'EZ1SxVWi2w7anYlkhrNp9x2bUbx3Gp2hHcvaeEIjqIHsGGnWwWuyhtXhqV45Rp11E6rafqEq/b+cUK8h6tUTpo3bSOqwfndhsVPf'
    '430GT6G+xr8DG+Bf0vvt2rIhvyZsq2Q7l3eJ7xsjL2VbEZFnv4udVx1paAkHsmAq1jexPBm5RlnA9BnE5cYn7bg+pHkvRolzHpCz'
    'jd1ppc4dnI99qqAWvQIPXyZYJ2BQBMunsaGANQIV5LWzJ11wOcKoUsc24kJuLe94pczs9E6gDhoX04MwcEYyqkNQnrq+GxEzAY+d'
    'kgWHpC5rD3WnXoAY8hlV/apTYb4dhqvPD2qIQ9PKXupOt36lJ5ZcD9Mr2gHia3CUUrfOMnK1tLzCHJFwMZ/j0iCGZHtStQLitbCk'
    'oBVQ9ZcXFL9gEQ02SYw9ZuxItyAlJfoQEBRAguYeioERfvdSARSVRtX/I1/MK69+Yx1eyC0ZWE66IvnG+5hgq0fQd9XVPqp2DhnT'
    'bZw+kzoYwHcKcjmxSjn+YNLnfkEfDy0e1Gypk3Oyr3JYGJOKozO36iUUxMTTrPVs8+mTkq9uWrp1PVKakZtAVaDTVh23sWdCL9x1'
    'O7+rjMCYCQ/TQLYHRgTUDgdJbcXf9AmE6Qz50Q700ZnSbUP4F3Ww7dWswmLO/8DGqtqgs11iTRbMD7oRpzRJeEDVjae8wHaA8y2q'
    'fAPGSoCcGiwUknVhJ+CkUmmroRFU8Ruh/w0n5Ll/agI0Yrh+HLFB3rJXiU2BzIDYDfSjFF/R5vgUdjDVEQCmjBgNywnFSGllk8iA'
    'Axj8GCetW6WwBRbLNMWmMRkD1e5KqFYl+aa1TRlnQqm4tQpiNKfa0BLL9MmiJA7u7q8PqCNr9tvmrda2TsuNVxQf19mBaSVbxvUl'
    'qW8/X6J/uoDarEPQ/zjWBlaaj4c9czyItaEjjRQkv5QztdrCs3ZVwr6v9lfthU6I0D2caf7D37QitRix9MwmXCU8OJZDEu0JhjNI'
    'ruF4OmfTq1EWFR0HNjhRkugTk6qYwgVoYZbmcg45AOZTWqwgXWV8UKuNTmCNQ7zry4++sJGOnYjCuBS10oucfA7sZskvQGKj/Eu4'
    'hRJdW1KAFD1rSeOJt2FTPR5vn2t0pN4LKepo+szrnTY8cgOY0AOAiusN2aLM4/YtZyuq7a+NdK/9l2LhZuygt+f4Vundy3hVIUJA'
    '7CV72tsp6KcyOu7VbVOjyumD4gR7/s7A/2i9124T3p7vknrd5akG3ykey8umXrwFB1hMEagJnikvSamd7m3uuKWZWPQc1IA9MHkO'
    'VRmZStsIJvBA1N1QjnAYvbRFcrVaVzMoFJW1kOTTcnkfz/T8yEuSavzyfc+e4nSjoeIg3O2CcATb076CTkzFvtU7ohOwwGbjFOKp'
    'SO8Bc6lJqrBmMGNxl91ty8jvVk5iYEmnwY9mjb7QLNcOA42I2zeHr4nTSzKAK2JdwzCdEaTL0JDFr+soN9KEacMXVnjbMrCsZCnD'
    'b9muGdnpMYhGvmBgbSaFlDyKyN9A+u6A9effXSMCx1kj/oyziqJVMzoKUOrOHlvw1Zx8Epmbd5HXSiW6b3RRKsYukDWrXoGjf/od'
    'p7hh7H5ra/2hPrGfQ6hAfSLiIthui1JSE4ynisIrTm+JZSNAPhMqfYDlOoCs0pYa4CNyywfiEs7KU93UJt42nGld4ek0KF6uL64I'
    '4RNvNSX//QFbxGmujcM38NyDte9eL2029SnnPoqBZEeca8THsjeZ9s/5n0/wYJyln5pmZA3d9Du729sDzyj1yXXJc7hEfdY0mmHF'
    'aZqzZuJpaHH4aIt6WpMGVzwn4Fa87uhUj7zy8IIUf0opkMvLmNGKgyyxySrw7CoHwzGGnrhPQ1flZyclCmw+OH5rrMqXG5s4k1dn'
    'p462zo5eVSxT3vLIYIhptAavDwD1Qfl3kbP2qfX1koIOnqYrb9/rfwDwAe170l/wpds8L4nPKPcdQ+bBtv0BCtx01bBtnwYFT5sK'
    'UhjnB8jgp21jvYf6ls7Hyh7l6OylvxeVznGbrdYvaTstnLC5llCsrfgpqAjscAkBEGxc5/1FS4P+NCDYetZssL4ilKLSuT/o9IVI'
    'taOci5VTB3ssQNfwFvW5nWyQ7eQ+JYTFcHKjf1tytux4PtFbUcwlaLmu0ShT6pjUNSu3FJW8iM+TBlgS4lwLWztvi7s0ubVN4v8t'
    'EFrsFOnMM+ncbFDFGpLuwqfbGARc/T+lBcRWoDuIMOVtfCvUAKLjE9fqMdqDJwiF/bK/6TmdvDDEGgpQrLnnau6aUMAx2w4G/qDx'
    'Q7eXpSnjCjuiJYrrtJqPl2Sg/UMGBPPsi/Vft5N/t8dy2v0z4UrOuqn/Drv4g1lFSvaV2JwqPEYJ7L0soTl4LYaOYAbaMF1MSzKb'
    '74lh49w/+kWXWbpZQzDbhl7YhXlPVWhsCijnDqm+MdsSy4xqZZOBkP1bdbjWHeROovKm7vPn0W7yhMZkUuNz08QWWHwG/fkT9Gea'
    'UWTd+M7fkp8wb3puQPC0ddHSFt5tDrH1Arq162bHiGCs1557ywVTAAGSnPbdfnvx21ZfBgC7Fn0zXoMS/LvSo/qLAZpkoa0WlojF'
    'q1Uxkvl+H/jrQn9EuRqunI2NtHL4kI5x7AvkUUuV3trIi9xBz7K0RulZyv7sznMp1LH9NRz/WsUWECxw9cSVHhj+C7G+XHtfuCV8'
    'uW1Qf+wggLl+53ZQD58WNanBjoXxvdjTdmAaK15skeAdS5twsTeIHPqfBVP683hHpx9RTQIw1/EdcrjYLtZPif/oM8hclfQnDJlQ'
    '8A5hzzt23vCCNE+wxHNfEgkmjXHBPKdG9Y/P6n/fLJoJ03P1786Tttc3zYXI5Jv4oxJ4IG0Mjtk5bsEkOTLpPJT/HMuS4OeRjmVd'
    'qPwrMPnni6mk0dEhShzboLLNDYvyHh7oGVkpYCCuSsdNaYokHwQ4dlB3k01IuKolCtna0vk01AEP1aEj2LBkXeQgidzFCoReuIpA'
    'rITNChPXZ9WQ2h20+HBkk353BgOHOzIbTuETsCi+xBxVYzoMXgdPS86bMjBvCt98yK69Pg8kvRMzKbCcPjOi9wLc1D7TP33+/Td9'
    'iDOoHYHR91o4uO2yPbynp/7mziB5LGCfp7plw6KASYTTQL7ktloKwal8D6UjhRbdLMOqHVM55QlpxiSjwqdMaZZ7vt1NfPiNo+I4'
    '25m0F+TjUF2qIUm9T8FL3OO77TWndrq3D5XXHW4+zw3bc+lTd3JdW25Z+hKcAqFAtdCkG+uqLbfNlo0VE7SRrxTvynuxCrvKOFzA'
    'cXZPgnWstEAb+6dYr14mp6lAeYK9QxfVU/nYOXepCfdDcxtNRpoPT2vDOY83LPxXOl91+YHuL4mewHVsDAVkOEHhX42Eo7jBWCbu'
    'CinqCE78qxaY6ORgJubtvV+trBXSvSG8QoOCGB9D7QGpx6rZTWrmBSVwYkHRPGh/MUU8WSXz/isLoBmSG+zRceIC2a7gpry24ezz'
    'NUOgvWU0MXF4VvOCeAWdNJ39Ic3gaC/HpsxkptxKqJdigSA3McEBk57agMeMOWECa3qBUwEEou9R2ReRziD1PGJMbEYqh9QL/eM5'
    'Bt03/RoAKzwKbT1TruZ7hNS+mZEBqgH8c4nrI/gveJSoza1L5V/Y+cNbaApeA8tqRQY61i6lGVBviHvJC7P8FBEk1oDiUd6HZsr7'
    '0FYbjlDGThnLG8eoUTunj8nM8Uv7gkAJSSTr2e3/HzaQdYYM+Mym1R0UrUablQhKzNOMjspP/0n2cUeDVUUMvXtHFuos1HVWYX2P'
    '/dDYhCp+tq6Ed76tD+geN6n5MqNjjKEGTAK5EFM7wRBck862b9Np8AqbCHQ/YVyDJEqba8Z2C0j7Dd+5K7ukN+7UPzZ4twV1aD8H'
    '+OdqVh4qu+trv6GFYScJLRiem1CoI+zqOeBcfQ01SCNpiQJjFcgG3XFdfZoTvSH9ebnBRERBg/yDLcM8+fRKfnG0BCaN3sjkUa4E'
    'TB+94B8t9pGhc3V6hV+tZnyZPwAAJQpBFJRDqUNDQk4H1qrokVscU9ZeDPOvhiEI4TuKCA0pPOGHR3G8OeoLJDqc6pvBOqf9SzrU'
    'IXsIKyQ4+0cyzwVyTnC4whjp6ZbeAqeWvKRalLfjgXU0vg5X+dPW3/i6bydCywSPrSypJVM10I9qpgaOUCof9IQN6mtIczOFXrsk'
    'wprPH+BVCsZXrQ7jHEKeO+l4Zy3vNNh8AjLdOIW1Krm7WlTXn7C6tLM4gmpwgoVwEdh3KNgBBK19khy3JvlukmQj3QbV6/B5qKQm'
    'oNCKgiS0CmtBsu9QMJqZuZhj3DTkiEXbdbP3RRnWQNspxQmIZxBEllLbUWf22J8eYLu5zTnawnUk42I/WDBs+vSjXowbkXiNQWox'
    'pe2UMoqgx4HHT9NYzqaxNjibmEER0QjOEuSrnDViu77CeT+wl+FEQQGgfM6UHqi/yQAUnMGTjigFOMtAaRyOXj6SYVUM7AwXOJN+'
    'ui5sh/Oz7iG7qDW5sN0Ddb9sAyWQlbNJeUMxtwz2NdJswQlL9QVFMI4/cvhScJLELscBvRY7hwSWr/VQu8poewcOGX5Lk9dX+a0R'
    'DHaXqXJ/r2cZyxRpikY029G6qESjR6boFbqXFjPVlBdnGPRIFaN/gi8k6SyCd7JM9iKD4uB7yYuS4AlynF4hbbrfG/FVCABQmv20'
    '2+zYJ24wCxmacX7AP76zRYMRwYJ+/aDPoGBu8fWwaSclNrfdlB2bDspxGMOZ4ikwkEOKsQSimCOwKQsQ+E4GH77PcLCUeOWwAXKh'
    'XKYuWbguh7WvVMclHGQP7+i1zMLq5s4WnzAQKknFc56wk70WHlm1dOz3quQTaOvumX9Bf7GAnBiPK5XRkpjeFt9cIYcYhsycMfnM'
    'yZn5lln5X+G46Uwvz2TqY+cBh0xn8qx0Q3/u2c6rlqPh0Rxi9y3tzl/84C/7wd+ZvoV06SwM9WA2lPDsk0D8010IdEBt5zBut0Gd'
    'rskon7BGSLX/6aHB1FO1mLQbasq/z53VXdvXKKUdF6ON6N5zLibsEuVVqyuV5IPSuFLsxJJySUhC2hkyxUw9dl2qqBGnWFu8iC43'
    '2iYvJ0rlfEZIMN0kl+KZyUD6HZgCXV1U19FhxfKC1R89/KrhU0UmRa+r2qJ+QfnjkWyMopLJAxpS6qet263PW3PeOpABPt9alVsj'
    'On6H8sw5wZEQsUfJFyg8jiKoKAkH5xinDAKjpK1zDPCnBX+CH41kEmf/VURvwd+aV57EtNLKgwVM9q42Xctw9q53wckcObk3bFq0'
    '/f8uUVC3zu/Pzu+585s67jyuSueBBqMescV333RPL5BNW9o8V/rw+U8Hp6c9KIwHx70u51jXyclk1gWNKjCXXo0UZu2OptyHndBu'
    'KdEiDi1bfbI2iZeqLLTDbIkYRayC06u2nxHXr8sXc+xFNbojIrCWZCqE7ruodiAGWNEKHqeHbrQGBwjrE3wO1vDfc3IM+kZuMRKq'
    'oczaQYueebsxIh0ZOpdMvdq1n3eeSqUexn+Ugk6wFe2bVOQ6v6WaQE4GLIvyp4NjROKdfDy+CGab0BT0rvNUTFugBiP2VQMytNuU'
    'vq1Cj1CGRg5TFKLFmRMsHDeZhIeK+kLToQ/3arFFPKUdsdBJvq9gxphVPLBL8X6eikei5yja7yMN/z8oNzzdL9GKvv9e5yUcDJqj'
    'mlWwft3f2rRhA4mHdC2TbIR6bsQz+tzJg/Tw7N0tMs4RsOcK31jO1x7LZ5JH5BHfSqL+5QtJ1G91F4nlvk5dygIl/EK1pUoFPdar'
    'yxZ6EOPZeMl2usexhMWl4lpSP9ZaYXpTYK0LsdaVVSZ/CpeMUFICpzNiH3RsUt3o3C/bQkBemzaS2u/KD3zfCrFjQj5aIAv7yEvn'
    'oHLUqIYoT5qd69PuWffIneqpO88Krx0f21LHxH4qGSR5QqHXCeVEUUXWp0XxYTiItPfk8GBNchNk3iOvR5thkIPHle8jvBb/NI12'
    'Wn/JfRmSGNB2ZiuSNCju2ZZiHhQeCZuIJht+CszfajOjzuMStZAHt0KEerIlgVNp9aDXa3RC76GMv7FyttXrFI1o2a25QC2gdulk'
    'm+fcSv6o+G+bbf7/aCjQdsmc4Yzoq+RzM3SxAbAg6aCDXdaTxiqvVZUrp8prU0Vbg4inMAVxRrrYySIlmX7W3Xik15E2FImQNR+P'
    'LdIeWDXSTW63qQl7oZIGL5eyWamZOremx1/RAV1BXRLlaZHY2A/oIp2j4guoJt+cjzd7LJbqXBCseEo3K7GI3OSTkvOmOClrRmQV'
    'yyh1Gw5ZxMdbbw2yj7ZlULec7n0cuxd9aO+MZglApyJ6YJRTq1wjt6CPtod3geHS23UfTSsSg9vuvvpuTUYRArd2/1VtNey+3Pn0'
    '6j6VyPFQSv7a1UgmQg/NSsZwW+nfcrCw+4oFqVslRSmM1mUpQbSWoWzScnsFFG9fHmSVRotT8FiBy0r467oR3mfQJMB2DWWqwPtn'
    'OmkYezgT6JzymBhSvbqnTubNJEp7xq3K8JapNGxmjXJeG9ljubJsrSA4HJmPUr0zBPvGg+T3pMWreCCCctKYrnKic2lDrDMxkZp9'
    'OKnVfP2kmq8Th4TXyUYhBYNyBRtGJHIWwKYFtU40UrKJAsJ4qiE9tpLRj1o2kgqSTFBV/uERPacbwkW8BCuHtfns+AwvU/nn7gry'
    'sixmouY07aWWChQBiZ7PRMmkg8O0nEJdMnux2fIG+v31TZCDkAx+CvXCztyNy0lJ54tKJiOdMvlKaDBNAq55qp7CkMr5Xbyr0wvZ'
    'qm2qKDnBLmecIuu/m1Vdu/6iZriGyq0cFLgKqLGvjwK9JhOe8pznsooySRbVK8AdobBJ941hT23mk9eWms2kTukowtMy/7ztRJIl'
    'B7bi7xIdx/hZslg4wHH8NkV//nqMY67t0Q//Jsu8JjvJvJkwNd1Y9vhVFFjbj2V1mPwAPOaOCVNSMSu0acilb+enhwcXyoKlkrwP'
    'eOl717PKZU6c+X1dyJgLTusmqopp7AA3a7oakYlBcyt3aqnf14Sd1ZOT1HK/Bz7Qtp/ckzDyjzLrJ8033/n9d576nYYr9AaBXkUT'
    'oVzeTBpgB4ZKGOxcquDig+dUX4nppFGnGzBT3IBpUqgbyZbv5+SATDgeYv6NemWGSY5C+GCT7+NFiDzjkkQGYHUDHUOO5TyYYhe8'
    'i0NbJhm8CbGbUKoA/zpP4653U9S/2ZQgFKZOMo+XEMzQi0EHEaaiEv2uTh0utKAUj7Q/mXduCp2uLL8O0pDRqAyIkKRUcR9BprRg'
    'qX7PaSsAargjZdLHklbTUbv2QfnwcVyu8YLS4ZEqdVE9WlLZkm4KvjYcUYnoM5hZ1HD/KuFTwP8QvXLHKS/9I5qhHbhcJxHrtv0c'
    'AuJbr4fGvvh7HDocab986tGj15pRoTCWXMcs0EkZfXdWAe6j5Q2Wl4Bxe6vHG5O/a/NLnZdCmyPsFbiBzzkakCBA4rW1GxbC9aZA'
    'tTTt1Nfrk4OeCy/DiulfWBwfGq/BoMX9ogn5tXRfHDsbBvDrZF8q1bPFWYOjmf1o0lZxNhKD/GRNHW/8Kh2+zuSgZ01P6gunirqc'
    '1I+2NpXES1D7ddLBeK/foQ4pUHqp7csn90tH5yorptR2jNr/RDk68CP3VefyaDeWmt3imRDdcOdGFGA+UuWLIgoFNvqKAotbVMs/'
    'Dr+iyzas86hXTY7UH62nfEw1EVtvUW/dPELPT2NcJt6sKcuEg4eQFdg0cP3dQT10J5UNwOMwXIWZTAMxagxTAqHfB3UHDneByIpC'
    '8pf+zqCW9aQyYBz6rq0adp4leqvNr/5JxjWiJz99hDOj+mdzQQIzcdKVwDmZLlD+UkpZzx1NN0kBvfk/EhVb2Bh7KxFIFM5Noazl'
    'ZFXFuqVNJn/dT6hpIlhiOUil57bHSv/RubT4hhj/k/rihptah3fF4HUTRn+i/4mYZAztHLhYs3BzShrPGP9oWhu/CoN1KoSJxZFQ'
    '8afTi8bbEc1Fv6CF23Ip9xQuWSJ+DeHysTpaB6bbVnS9c/jh0dkXDr4Of7OkqyTGHlIGvjnXCt77A5TmW9Vp2Nzi2x1zvw0/8+3N'
    '/Eq9wFVqFzb3lQnUWi3orF40cByPDJduLuHfjX3GXqjzpNxBXl29zTq5aFiHc1Hvtq/1uXBVGpiiX35eY8736OtPg2zywQ2sTOz+'
    'y4fxHQf6X4Hx33aNYn9inGZe2SyLNYMGfe47jTrJV9agjWUzVU25BqzBqouwoGuxD8DRGuDRhqM5sBVyQjUaQ6hd6tQvfkISs8lq'
    '+7JW+wLv7QedXFn/0NeLOWeVvnkmCFS1YapA3NnHY9HD1K3KvpLlCO4kG1Yp8RS+HbyWmrhWdat2q7mJp5kvySGaAJKkIHehPw9b'
    'UBI4UrebomSu2+Ek+jEuSzn6eJq+PevusxHRQAwzlUhdMn4wVFYVg5bUNUoEQz7JFQYOTO2YxXnKWJT9U/lMUzKIyOWp7OXOK+Lw'
    '7C9fIXJYZKgSCWDH4d4Cd2pZ2PwSDeikTvBEn3IkrJF/6E0bhpXpL5JdDdQiLoyg12F2v/dL7+Dde3DN3n73N31Qj4OldLIIRfVr'
    'tqnrFoSTpwLTvQZLVdZjYBdwZ5Gym5btH3at9jsYZc75tblr00/eJldhem5xEzADqhLTxATGNnoFHcySTPM1wVxVMgNvOYU9Dov0'
    '4GyhlLKbTrFEhTHveBTiF6l5o223dzjm7R+gVUS+6cCwF+JwiKT3spBKyoKsW0/MuA2zv9yQlQEC2GYGO6/Mb2rZPuRQBUfysS1F'
    'cwnNILb3lwZM6793/vGo5+UpE4dIuyupksGtw2b/Tppuh6Zsh7rAI+mipfSTLl0O5QGndwJnvZ2VeLgj69Hln+ITjsyh5hqyPXWP'
    'q14qiOPC6WLtCgOzPTVsX311fYclfBOOYlbQmv1lTqMKa8myWVdFvjbUUhSji/G5+0agBlhJRxWDny/wc18raIUgVZBvsuOAmYaC'
    'WjpyyyqybSquZSe3eHN/zSz/zSGupmynN3wvHIcvqkm2xBrKPbJJ0J2AlHWOwgVo+lU12hZU11vKj8AsQXVvqyw7+1WeBx7PooOY'
    'lGjXk7sURf5pCKbTKDGyC2TJF7ZZCgoLaDroaHrxCtRRZEiiIx3nLPR69juaMvjWHD3VHU0G5q3hRXbaG5typ7TjTLojusqyJDwx'
    'ZTsTBh6Jo/fltPQWPHgRwngndFdAU0l/wrVYxCzRnoabGdO8UlsKU/gLhFciywdy6qP7K5q+UiSnOtYyB0X0bmaNjnXJzG6yIlSZ'
    'XDR1YhMxqbbjSTHh7oOG2ymVrjn1k+GGwlroyKJNszer8Zhss/li2sRhnJb3uJU1ZWibGvh3K6g+BWA8u5z7eV55X8UuS2V+EKsw'
    'dTDk+JRZNpWsQPS9j+Idsd6FgmnIaEhHClL+eOlTDeimiFOqrIUg99CjaGTtNC9WpAl5SNj6dUvLWfpcs1kuM8WqpnTQRjRTsS0E'
    '7nkU+NHLJlfJLPhm4cYdSoZApRLveLahKNxx6EdqOx+UM9e9qyvdJ0UpNyHNOPLCM2G88AXkd2d08cbhwemaXmIoa5vV3eOwkXjt'
    '9s3BGuw1wQt6nlJ+OW0Dqlm7dG4dQzPJI4uEfjSWUGQRHCh4vMecJfMpudbhk3rpmYJSN04HjT2DO+y7tPczDjp4qZw3JXER2S4r'
    'K0kRbFkjxGLLNRuClFm1zyn0SlffCttrCIbOx5JSzpW+g343tKW2emG0fA00N/1stMUPjD+6Noh+LyhtZc43ReMO6WqLS0IWQNSJ'
    '3CRNZ77+1dAeLkSrDPrkKQwtO2miT/CsUFYkOHqd6eW4JkJa460luHhuEbkVfUen7k0B3Q37U50qPB5tluPajW8NoZodleORvH0X'
    'kY4hfaDl5Al0eN79ufcUQnSkrRpUv2lIIWcX6cXBUQ/a+Hnv/PwAR/f73VMyCX4NYA6UCEyM1nTdRITn0O8oKIe6AjN4SZ6K8LYZ'
    'qZAGufeT/XfaUXdJq1ii8KDc5DiGuKV7y5QbNV2KLTfOsQWHZJfsmm40wJ2TIta2PdEpuKfPQY43hHAz1bf6WUdR7G8cO8RWFboH'
    'UPK9yh3joUOxlWRorfMSfOFysYdEhbqo4eumFA59H6jHxq6j1eKAnv7yb1GCzvi1vht2cZCxr0kibEjF4BsM/WI8DDHmiW5FWvHO'
    '+jJObgTeNtaX1DkTBn5sbCNXpIj76oYvbabgV3N5/DPi7MwekZ2W/m1FZ3JNKvFFwgAxxkq4osI+R7/TyFsKB+HKp3pJfZUCVT9q'
    'baNhHVp1dLyz5qvWQR1D4lrFs0npauCBVkOxtVuROZFI1i92Omu649vf9eLYePgKLW9S2CamqVnPjgdBaWta5uSZdyNxm9gWsOzc'
    'oaBS27IFrtMkitEniWRGGlkwKsv7wzOUdVqVWmXreEzN9WdE1nu1/2K3xU9GINEkpp4JkQJ6FdkK2fl9ka2wZdVWSxnBzakOOSgF'
    'io5DXYbQecBw8Tj9PIF2HCqom9psQacfXh8cs5Yp/NWmrYASv8oy13Tq9hXWuXWD+fdsdLUpbTxFU5F1ckU977XGFVdBNqMIMKPr'
    'Hjt3CtGOAXfFRjius3dRNVw6JHeRq3TmTStFEj48NKYme6StEJyJ8FLSnVQYiNcsLRW2tGYVBYddG4P/AqX+eFE='
)
_SOURCES_B64['m4_lorma'] = (
    'eNrVfQt320aS7l/BKMdnAZuiJMdOZrhRNrRM27rRy5LsJEtxcSASlGCRIE1QluRM7m/fr6r6DVCS594991zPxCaA7uru6urqenV1'
    'f62YzmeLZTSrzkr1s7qzvz9Vs9I8XExm5+Zhmi0vzcOymObmYZGVo9nUgru8XhYT8zgvhlcTW/hrMR8X9GzelNfT+V2UVVE5p7ff'
    'Rb3yS/QlW1RRtsijWTm5i8azRTSZDbNJtMyrZVFetKPDMvo1u7iY5NEkz77k0fIyn0bZZFbm7bPydfe0m77ePY62Mc52Xn4pFrOy'
    'fZEv47O1o53jVH8/W2tFB6iSnJX7h697e/fVMQVqlY7+uL/O0R9OlcMPp/e1oj5ThbO1jSse4cbNbHGFQW/Mh4t0+iKdzBbT7Gwt'
    'IWxJCwfd/V76bvfgFGDP1ib5RTZZv/icTWf52VoUfQf8TUb5IiqzaR6NF9nFNC9BAWNCWnR0mVX5+la0Pxvlk+iFKqshn/5x1LNA'
    'UwBNp5f4T0OeUq10eTfPo5tFsVzmZVTg/8sqGs7KcXHRFoI6K0/2D3/tpae9E+rjm2xS5VHw57vodHGd4ysm+C5aXJfRchYNL/Ph'
    'FfdziJaivBzRW/wD8hk3o5BbAnqiber31tla56wk+F4PqCnuVq/3Go8vnp+Vxx8O0tPj7u6B+cxv3PLHvZMP+z31GDX9+Y7pPbqe'
    'R5n0fT4rgOvxYgbqLKNsuczwehTl2WJSYEowzLOSW00PD9LXvY/N2DG4wadZtFxkQPKsjEb5l+jzdb4o8iqKgaSSMIVFU2AxzbCG'
    'JsUoWxazshVNsmoZ5fPZ8JI+Xlf5KJHF9no2vGZyWOTzRV7hF9fAGjrcST/u9n7j2c+ulzOa8NpozaeYh0gThSazdcCaR1+K/Aaj'
    'ziZ3VVElRNGXeTYS2l5m4BFr+p08RVjmZ2vj6wl+n5Xvet3X6c67Dwe/nqAPW0BTd3fPe/Hmw95eut/93b78e62Dhn9449FtcMV0'
    'r0dzvrX5/EXThC5nV3lZRXPMFuDcZItRNM+qqgVCH06uR1iYQp+TWUW/T3pHUuWsPDo83Ns9eMstTvOsbMAgo1B9m31BEwzq8rq8'
    '0u0yTmj6CMmNDWEmP5z00qPjw/+1mjS/E6rJzsExR9Et/psvZp/yoZBHtcwWWLNgwtRCMQIZFMs7oLy3f9Q77p5+OCaq32xvvhSq'
    '6Y6y+ZK4xN7h8X43/dg93u0q7gM8TYOBfmdfx3szVFifFx0Qb1npvixmQnaM5Ul2ly/Qp8nsJl8kgoD55Loy1Z+1aG1l80S3T+x0'
    '6wf91N07etflN01oYIKgBRCd30UENoor7CyyuOaXWbQRLQzg18eHR2DJauwPgkI16srLfP1FtOrPd1EmyKONa1HSTC6yJZgLzV/K'
    '9bdW1P/OmbOwNs3KKc0wXp2V7z/0jnd7JykmLz05BaFgaRA9yhv6SnvW900tgLiWxRcwlCqbzicYG00JMZk7/lUtsbDjC2wSUZUv'
    'ZY+eXoOtTLLFhSyRqJphn1F8Cah81z1+nR703vptO++Puwdvib7i71vRi83E7c4lLbcSe4/0aTjjHYw5DfhYNSmGud7IXu0/fwm2'
    'V65z5yYFGF5cXRVzoekvNIDlbN6Kbi4LdHea3UXneXRdTrLznIa5wN9fsnKJDveODnfeETsBtf/W2337Dntyb6f7h5AB3nWP9z8c'
    'pW+OuztmVbw9Jm61t0uo3qJCwlr2d3eOD9NX3dOddzX29J0s9DpriUbXC2YrZj57H7t7ASyid1r2O+96O78eHcrmX1v830UXi2xU'
    'EIO32xFAtoDUfIRhg5IyWilvX0WnL7Afdj/20t5HzBHTDSHh+UtaDW+Dty9p7zw52cWutdM9St8dfjhmrrzZfukxHaCetvIqo+mD'
    'oAgCms3BPOkn9hvsPRX1gcqV+e1SNkQi5t1yjK2sHOadKKfJw4QDJZir2Q1tYcPLGTYrtQmCUR0epb+i/R/PygOs3PTt8S7t633I'
    'BCCJnHaaTexAeBxWk4oeX/qPW+rzqJpy4fbm8/DFy+DFVjIAEf/xCk2le939V6+7hIE+StLX5y/pb/y1Rc/P25sDwchNXlxcivRF'
    'zHudyTamTfAZ7YIJ2APNC6Qcwcn1AmJvNZwtsMipLG9i6SlkmQNG+PO/b6oPv24x7b1Uj6+YNn986UhfLGco3kAz+4KlKCsYKUnJ'
    '2xWf/11emkXxXJ4bWMwLt3pA+UoGq5PX9/KlTmGqnTmWAol2jvhWEMl0RE4kaQPoFdkGdIX+rjNi1eJqNUiRxLfbSoDeOTw47R2c'
    'qtHaka9HWw6fzxVfBLmh8at8vmRGF2zHBA8i6TS7ykfFooqVMN+K8ltwo3R2tU3rE82KxtSusARjkkDxqpy3G97qwe+wMA0dKANt'
    'UNdBgGujfByNi3KUon8pKVXxYjZbYsQk5ydqLi9JFgdVDuSRek2lSErn0qqY/oZ+p3Noei3+RYAAj2DLb6qGEd5kkytuLHGq0x9Q'
    'EysZKGYrBWV0r9rZfA5BPgY8arH9Cawptu3zIBKpCrCTvIypFov1mw7IRb68XpSsXtkRtytolvFVfrc9YfnIKUif+5uDM2YzBoWi'
    'x8zvYj0ipdq0oZpixjPZP7D9TEEC1DXIJzMRALLhkOgh01rW8jJbipwOzQpQcoJB7M5RoRTo+pz0rdZXlPPrZaMieLY2+L8+bZBC'
    'dafW/OnjvtPnfPXHhqGtPUgEakJWzL/bIX8GZaobpg+VY2osxVZhxjieXaOL/gr4BnTV5sPFHe1F2wZamwXWOLHfGa+uIrwGAWUZ'
    'YJDlW7s5C+IAKsAYwIDhXef2raN7k+BN+oz+tFzcBdVviuVlNMOCi1fi2+tokpA2MG6YNq9ZKtwmthSPE6WD2+9CvGb66E9+y4ul'
    'x/+AnQbw7xkScOl+3Y4cA8VMbBs8783YYyrQHEcP2ucuXORh9kL0Q5V5FqlKJ5hvEGQ2kVl0qSLokII7Z6ua84JBGv4EgMaWBTZE'
    'vVBwXn04eL0Hpa97Sturvw3UmciAVW7aFouv+SiFIak9v5rwzOQwNjTC9OhE92I1HHTVra96y1MzWxpgvBNWsVNSIwayLtTBN+j/'
    'wWz5hrDQWyxmi7ihPYYoMxp12aLiWLJ+dAwQs+slhh/F3dEI8iR+JrLt622VSkbngDMBCkCsbrcU+q1d0Me/ay8MWFBgk7NUtgrW'
    '/cM/mK2w0UWxsyBq9rkkwJBrFVxnGvVxwbAUZEaG6a2PClg7mzDBRtBgKw0GXqt5/7j1YFlki81u/Exvqc8aNlTm3KA6GUbSQCgF'
    'aYU524sjErpRqgkP1KiDhaM/EsdivZwthpf+U7ss2YRd1l63x9cla+wQ41HizVkp+it/Jkt51XYMhaqy84ZsMlQ2dbYIMs/1Pu7u'
    'kNIsgKAEQRUmfeZ6lBHzGasP9NwuqjT7AuWCrD9xEtGSJ34/v+aRkxbZ3ScBXoC2NYfV0M5KfE5fK1uwwB2D6S+1Dnqy093r0UpQ'
    'oM5KKTTNymvMUJNEKx1mFEurxFn+GY3n0EXJpDiExYu/KpCM/5PT7vFperrLNljyQbTpLyI0Vxc96ZFaXtdOn0bf/7DJCvxK07EP'
    'hYzIbEpdulRbb4pxET8SIovudjd2WA5hYHFOjTo7L2lx4kLR26yYrY9SVnAJma/A76t8nhYwrQ7w+QiWCO/jPBuZjx8Pd7qv0pPd'
    '/+w5Bb4A3edpBR4rhfQkVfNJQSoCsca+flS7yT8jrsUfLVD5QqbJYsSfpKfyGh3Tr6WPsm3o/m6rwh1f8aNaJGUS0OqS7E0wpAMM'
    '9s4RW19JPp/cZHcQtrPqSqxx2NAvWIsg2wG9EJ2wGGm97+3hHqGIhkX2IgxK3p30yI7y519YpmTJohETU/GG3zGMTb7zUiHPlPE1'
    '1AU2gva5GBEsaqbPNV0xXjXej/lLiwonA3QFVrbYqdHH+4Gy3f9GmoY2ucPEAgzEjDXgoJopU/IMy6xSdpMhMHeeC3dhD4A3RgyR'
    'LU4itpFZpT5UJb42oaM+5iE4r6d9EqbT65LUaBlY8kjkCCQtv5G0VkNJ4mtEwwAYFwpkMNudNkgpHioQ2WKB7kEhx4/sLpa21bf5'
    'bDbBR+oBryw8YmkRFlQ7iU+8T6pO9GRknDT/FAB4809u3TGukhMgetLegpgL6wH/3sTv7FZK09OTJ9rWyVBA9wzDM7mS4vUEZioz'
    'UEVO1GOMByYsqtuiQbapyRhvMFRpU5Xgb9ktfdoihgnOSdUtuhIYyQlMwoRIbp6UbHkTnuxXiv25r4mk/vwrkLwanERRzOaw/WK4'
    'mK2/2frlR2XgS5hpuBCFdVinjvVSqUk+x3pkDQtdstI7v2VrGl6vs6FWU4w2XPQfclO1jANp4Mv+GoKHEGJd7os+lRpEPztdCcjS'
    '62O9akNhNUz6x6xX+6EudLmV3JFJCcdTZgo6M6cdhjxx5P2QTVxVEk5PIJUBjj46rjz5Ts053x3PnnwXD5gwcSpg7HK8eyohSf6Z'
    'FOdtL+CgKKs5fCREmYhsELUD73IYhGCI1YoIBHUaV2zkOxAU1aNd0YPbprcpyWysXKUUh0DiXKytHhBHRf4MxEX9dVQHiZfXAMVA'
    'CXxMfyXSX/lW9RvAD5jzKKjSXRYK8gX0KnRSqsamCK/Ovf0W2w3TAtzeBdDeo19v33f3Z703s8VOdl1lk739NvcKS5P3AghuRg9o'
    'KX0qJXDYeBmiNmlOi4rMoClsbprj06LCoyjkqgPaRuCUJrT1B4ndaajO3zjmYJoSIbXFZI6V7XkWCmsll809n57nLBE4JhKnHb17'
    '4LfDiLRqJW1UysQKhquqUm9EAiIW6MJLhFSv4bYgekMlryg5oOJw3LawO/QkEW5WawFcYtNTj4+xFUHi1SpS2Pkbcv4fHJ5GxJ7u'
    '1Fhaeigd9BdaU7VceK30Oy8HqgOYfWUEshIhTYWV7wLZDJ6wA3i4O2KaEYmQBtUEiNAFWCyrmIIW8j09+CmQDBtxwXA33I7QuBG1'
    'BFVtTQlMiqCII95DPu1od0x64k1WaXlHRCmrpSEc6hr+RchSDAeTCk9ksSS3ApqJnXYIALuI2cqLRVXAy4j99/xaRDfUNy4FJbPJ'
    'EPLPUJ0qL0RHtFxBVH118DprIrqGxaZXG9Dtg8HSxbubZa7BetMdlF2NoPWf712fej5O8U3ZaNjpL1gVIb9gelZ0TepSAyBwJAp4'
    'uSFy0Di/mV1DJsLMKHeKJiu/720il3SOpZAQgXtjdr4pNGEs6UU2N6peXMfbug8jaWfnVZyIGGUNIRrQz76NsYmeAxq6B51wxwPk'
    'k/aLcfLvHr14bIGAQI6AYp6PWEZUfXF4oW7ynrY60exKTR9tFYwNipbYF8JZzmLR5vldm0g9VmrGnMgE7+bZAvsu4h8qg164vkDt'
    'BXqXkrdah0BRIwfwIJ6QqgpPcTxvI1YwB8RoFUC09Vqi7pR2608sKGue97cGobmHmTZZPVA+aacs8aSp8HcGT1J8+/l4H/vCZQH/'
    'LBzaxJieCCZj1csNRG78AFFIeiDCMfUUUV8LJVmCHlLtSkwNVisRKkvzSb04h7BxpfRy9oigdpV/TrFRyAvSt+A1T62kRFCGy1sr'
    'm2IKOHIMvID6ERsO2+J+mcBERaEFWAWigeB3j7kmhTktE5FiGZJPuqHVjLsTxU+qpGOwI81w7cR3jlh368/yPTSSNy0MW+kJrw7E'
    'fF6I26102Kfuiu6EqdXQk/NFnl1Zj9J5Nrw6J2O6THXMNnTS8qxv8IRkZ3gD8UHiljKfU6toDtKgKN5hTkaJfEwSvWYbkchb0ttW'
    'tGAm4lg0wAOnEAOrtjR5yvSs+mHoWL7dytobLWZzRe9OlxOtisPDhamVoimK5ilHz5BMfNpyYIthrhXdtke0IhwN/XxypZYc02Xl'
    'Kz9i8StkCac5R5oBAUw5fpxKMMmEckR9xLDWoIVt/JesdhbiKxXUwxHqbXHci/7NvCNhF4oL4FabLh1Daox2MdIW7dBQsqDVYNNY'
    'bgsEx2llXSQuNOrL7Tf1RQ2Cdo8yHce3yZkKdtmH/s+2UhMHxDthdoFO2d2PaeXf4DG8KXU5kId8FmvzjLFvGetidp4bky2Cc+EW'
    'j22kAf1DWPh+lcBGf2/HW4jK+nsCUhbS2DYcnppAX6APXEiUgd0W43ARcVeSxG4Zgo+xW9mS7bYUb8tHv5IgxWzFtgOw11zQfru+'
    'xV48C9z5YDqgjB8K5qgYj5tAYlu3YMI93fA+f+a+VJH2pfCLDoWdgSdwv1lzftL+fkwaBmw7DFmapw1cMUwq2eK31A4iwqC8qnGD'
    '/bb/IXuB1+polougwBKvwwhVgbM1HTLCunUqEZ/xXouLaYLZoX3TCYkx28IeCeE7e3XHqKFtbDscwCr7Lp0WaA/zYhLvYVvc2dOI'
    'VoGm1iSo3LpiwBGrSmBUUXDRfhDvW3PxYpPluJYMG0Ks6jXxEumE1gencG8WMHHtABd7mHDb2RVrn+1t1BBW0oSsDZmwUancCrv5'
    'cA+YE5GXKq4S3XbQbimSGXzT7js2o3jvNDpDO5a18YRGUAPZMdKsgtdkDavDU71yjDqrJlS1/UxV+n85oV5D1KtHTBu3kdRhfXJh'
    'cVDfw30GT6G+xp+ADfAv6f1mbdlQXBO2VbKdy7vEj42Rl7KtiMiz08XOq1waWsKBLJiK9U0sT0auURYw7YM4W/uso/KHNO/FKHH8'
    'ATnb2J1W6tzB+dinCmrRK/CIZYJ1AgZFsHwaGwpYI1BBUTvb0gWXI4wq5baR+HhreccrZWandwJ10LiY7oUBH8moDkFF6vphRMwE'
    'PHZKFhySuqw91J16AWLIZ1T1q06F+XYYrvYf1BCHppW91J1u/UpPLIUepue0A8QX4Chz3TrLyNXS8grjIuFiPselQQzJ9qRqBcRr'
    'YUlBK6DqL8/ocIZFNNgkMfaYsSPdgpSUaCcgKIAEzW0UAyP84YUCKCqNqv81X8wqr35jHV7ILRlYTroixcb7mGCrR9B31dU+qnb2'
    'GNNteJ9JHQzgOwW5nFilnHgw6XO/oI97Fg9qtpTnnOyrfOiNScXRmVv1Egpi4mnWerbZ+6Tkq8uWbl2PlGbkMlAVyNuqD6Vsm3Ml'
    '7rqd3VRGYMyEh2kgmwMjAuqAg6S24i/7BMJ0huJoB9p1pnTbEP5pHWz7uqywmPOv2FhVG+TbJdZkwfykG3FKk4QHVF16ygtsB/Bv'
    'UeVLMFYC5NRgoZCsC1sBJ5VKGw2NoIrfCP1vOKHI/SNzQCNG6Mc+G+Qte5XDLpAZcHYD/ZhLrGjz4RsOMNUnAEwZMRrOJ3QATCub'
    'RAZ8gME/wKV1qxS2wGKZptg0JmOg2l0J1fWcYtPapowzoVTcWgUxmiNtaIll+mRREgd399d71JEV+23zVmtbp+XGK4rddXZgWsmW'
    'cd0m9e3nNvrFBdRmHYL+x2dtYKX5sNcz7kGsDX2MSkHySzlTqy08K1cl7Ptqf9VR6IQI3cNS8x/+phWpxYilZzbhKuHBsRySaE8w'
    'nEFyDSfSOZuej7IIJ6ssbHCiJNEek6qYIgRoYZbmcgY54JKOWpGhQRsf1GojD6wJiHdj+dEXNtJxEFF4LkWt9CKnmAO7WfILkNgo'
    'vw23UKJrSwqQosuWNJ54GzbV4/H2uUZH6j2Too6mz7zeacMjN4AJIwCouN6QLco8bt9ytqLa/tpI9zp+KRZuxgF6205sld69TFQV'
    'TgiIvWRbRzsF/VRGx+26bWpUOX1QnGDb3xn4H6332m3C2/NdUq+HPNXgO8VjednUizfgADiqPIEVQZUipXa6vb7llmZi0XNQA3bP'
    '5DlUZWQqbSOYIAJRd0MFwmH00hbJ1WpdlVAoKmshyafz5V1c6vmRlyTV+OX7nj3F6UZDxUG42wXHEWxP+wo6MRX7Vu+IzoEFNhun'
    'EE9Feg+YS01ShTWDGYu77K5aRn63chIDSzoNcTQr9IVmuXYYaETcvnG+Jk4vyQCuiHUFw3RGkC5DQxa/rqPcSBOmDV9Y4W3LwLKS'
    'pQy/ZbtmZKeHIBr5goG1mRRSiiiieAPpuwPWn393jQgcZ434M84qilbNyBWg1J1ttuCrOfksMjfvIq+USnTXGKJUjF0gK1a9Akf/'
    '9DtOccPY/dZWxkN95jiHUIH6TMRFsN0WpaQmGE8VRVSc3hLnjQDZJzT3Ac5XAWSVdq4BPiC3vCcu4aw81U1t4m0jmNYVno6C4vPV'
    'xRUhfOatZs5/v8cWcZRr4/AlIvdg7bvTS5tNfSq4j85AciDOBc7HcjSZjs/5j0dEMJbp56YZWUE3/c7zzc2BZ5T67IbkOVyiPmsa'
    'zbDiNM1ZM/E0tDh8sEU9rUlDKJ5z4Fai7sirR1F5eEGKP+VLyOVlzGiFI0tssgo8h8rBcIyhJ+7T0FX5OUiJDjbvHrwxVuWztXX4'
    '5JXv1NHWOdCrimXKWx4ZDDGN1uD1HqDeq/guCtY+srFeUtDB0/Ta2/f67wF8QPue9Bd86SrP58RnVPiOIfNg234PBW563bBtHwUF'
    'j5oK0jHO95DBj9rGeg/1LZ2NlT3K0dnn/l40d9xttlp/Tttp4RybawnF2oqfg4rADpcQAMHGddJftDTozwOCrWfNHtZXhFJUOrEJ'
    'eV+IVDsquFgFdXDEAnQNb1Gf2MkG2U7uUkJYjCA3+rclvmUn8oneimIuh5brGo0ypY5JXbNyS1HJi/gkaYAlR5xrx9ZO2hIuTWFt'
    'k/h/C4QWB0U680w6NxtUsYaku4jpNgYBV/9PaQGxFegGIsz8Kr4SagDRsce1eoj2EAlCx3453vSEPC8MsYYCFGvuuZq7JhTwmW0H'
    'A19p/NDtZWnKuMKOaIniIq1m4yUZaL/KgGCefbb662byr/ZYvN0fCVfi66b+O+ziK7OKlOwrsfEqPEQJHL0sR3PwWgwdwQy0YbqY'
    'zslsvi2GjRPf9Ysus3SzgmA2Db1wCPO2qtDYFFDOHVJ9Y7YllhnVyjoDIfu36nCtO8gMReVN3adPo+fJIxqTSY1PTBMbYPEZ9OfP'
    '0J9pRpF14wd/S37EvOm5AcHT1kVLW3i3cWLrBXRl182WEcFYrz3xlgumAAIkBe27/fbOb1t9GQDsWvTNeA1K8CelR/UXAzTJQlvt'
    'WCIWr1bFSOb7NPDXhf6IcjVcORsbaeWIIR3D7QvkUUuV3tooitxBz3JujdJlyvHszvNcqGPzWzj+hTpbQLDA1RNXemD4z8T6cuF9'
    '4Zbw5apB/bGDAOb6natB/fi0qEkNdiyM79m2tgPTWPFigwTvWNpEiL1B5ND/LJjSn8dbOv2IahKAuY4fkMPFnmP9zPEffQaZq5L+'
    'hCETCt7h2POWnTe8IM0TLPHEl0SCSWNcMM+pUf3Ds/o/N4tmwvRc/avzpO31TXMhMvk6/qgEHkgbAzc7n1swGZxMOg8VP8eyJPh5'
    'pM+yLlT+FZj8kRhJ0ujoI0p8tkHl0hsW8ztEoGdkpYCBuJo7YUpTJPkgwLGDustsQsJVLVHIxobOp6EcPFSHXLBhybrIQRK5ixUI'
    'vQgVgVgJmxUmrs+qIbU7aLFzZJ1+dwYDhzsyG04RE7AobmM+VWM6DF6HSEvOmzIwbwrffMihvT4PJL0TMymwnD4zorcD3NQ+0z99'
    '/v037cQZ1Fxg9L12HNx22Trv6am/vjVIHjqwz1PdsseigEkcp4F8yW21FIJT+R5KRwotulmGVXNTOeUJacYko45PmdIs93z/PPHh'
    'N46Kz9mW0l6Qj0N1qYYk9T4FL3Hdd5srvHa6t/eV1x1u9ueG7bn0qTu5qi23LH0JvEAoUC006ca6astts2XPignaKFaKd+XtWB27'
    'yvi4gBPsngTrWGmB9uyfYr16mRylAuUR9g5dVE/lQ37uuSbc981tNBlp3j+uDccfb1j47+RfdfmB7i+JnsB1bAwFZDhB4d+NhKO4'
    'wVgm7hz59whO/LsWmMhzUIp5e/t3K2uFdG8Ir9CgIMbHUHtA6rFqdp2aeUYJnFhQNA86XkwRT1bJvP/OAmiG5Abb5E5cINsVwpRX'
    'Npx9uWAItLeMJuYcntW8IF5BJ03Lr9IMXHs5NmUmMxVWQr0UCwSFiQkOmPTUBjxmzAkTWNELeAVwEH2byj6LdAappxFjYj1SOaSe'
    '6R9PMei+6dcAWOFRaOuZCjXfJqT2zYwMUA3gn8q5PoL/jEeJ2ty6VP6Ngz+8habgNbCsVmSgY+1SmgH1hriXvDDLTxFBYg0oHuW9'
    'b6a892214QhlbM1jeeMYNWp++pjMHL+1TwmUkESymt3+/2EDWWXIQMxsWt1A0Wq0WYmgxDzN6Kj89J9kH3c0WFXE0LvnslC+UDdY'
    'hfU9jkNjE6rE2boS3smmdtA9bFLzZUbHGEMNmARyIaa2giG4Jp1N36bTEBU2Eeh+wrgGSZQ214ztFpD2G75zV56T3rhV/9gQ3RbU'
    'of0c4J+qWbmv7HNf+w0tDFtJaMHwwoRCHeG5ngPO1ddQgzSSligwVoFs0B1X1ac50RvSn2drTER0aJB/sGWYJ59eyS8+LTGjvKQd'
    'NXmUKwHTRy/4R4tjZMivTq/wq9WML/MHAKBE4RAF5VDq0JCQ04G1KnrkFseUkhjD/KthCEL4jiJCQwo9/IgojtdHfYFEzqm+Gazj'
    '7V+SU4fsIayQwPePZJ4L5Jzg4wpjpKdbegucWvKSalHejnvW0fgiXOWPW3/ji76dCC0TPLSypJZM1UA/qpkaOEKpfNATNqivIc3N'
    'FHrtkghrPr2HVykY37Q6THAIRe6k462VvNNg8xHIdM8prFTJ3dWiuv6I1UUejPT48NQLH017f/Qa40lx9AZ+LhwqgRWIjkSA7HXk'
    'khP8JN9NnnAk5aB6HfaaSgIDOoBRkBwnGXk5eWiuCRXzG/NJOA05YgF41RzfKvMbVkBKpwkkfgiCzVxbW0sbHEAPsPBc5Xwmww03'
    '42I/WTBsIPXPxphgI4ktg2xjSgcograHnWCaxuLBxgrinGMGRURJ8DjIV/FIYlM/R1QAsJfB76AAUEprSiLUX2cACs7gUY5MAc6S'
    'UhqHo5ePZH4VMzzDBc6kn34AMXI2g9JuOQBMvOFmHBLps+Rk0zpPJpkkdZrnDs/0XRRD6PmUEJo/oZEioVjzaKQn7o4+jFp8YFm1'
    'kHiHxNXZU0unDwQymp5u18MMXd/WcDb5prquyV+dJGEQ64qr6m+2o310f8CpA+nghx6yIqSgmI/1/C5vxLiKGoxEtMXx4mKy9JeY'
    'OkYLx5LJuIucHxeUmOqbMW4YgYNx9CyIqqB+hmhzMZVdPIhlD3yfqrS4ogkHNLg1XdK4RQUfrX4JR6AlqlScft+ilAi2oIQr++T8'
    'qy6LMZ3xl4N6QGxBvJzsVPHNAtsVBWlmfIgjaUe/KRbHCJdgTbUkeH3pQ33Ef/YD7fZ+ut1vN1Eusj/MM8prSKGY2u3DJBys030/'
    'bkUhZl+F7FKdgRvMysnW9zhatSmalT/j6D9SkxR0/p1zdOPqA/ahKwpTp7urYmTydINzfQVu+MA3ULVJ2JV3LfbL4c1tW7cRcV71'
    'DquIcWwTtCMvVhfb9m5CxfkPjpxHRL9fSYv3Csbd6GviAIQlSwAexYBiIQR/NMDjCJ49Sm5mCEKqCdmMXFrQjbzCsXQV2pjfZsMl'
    'ZYldNh7NjmV4nKv531XXqK3dxKCAXnLGaz5uigZhB9YD/RrFx2Azu8l/nersAUh5gE2Es8NT8nYOr5TQHiQqltyjyLHWXhkffJ6R'
    'coW9iXHY0ts2HZtdFFm5/KbwYQLGUYtV7qtcrOqaIk2Hwo1WsOpwuOEjqbjIGJB30Ns9AsptUW0uHrwnXXMRvFPDJWux/Aq+1y4N'
    'CL7PWSCiBgWBTodxa4ffXVmVAQC+5iXtNsdeS6QiRb+lPDmWa/oBcQ2GXgf4q3sDuwXy4tuBk7pD3Gez6QoDimaCx5zvKqHT25z3'
    'AXwuivVNCkTFWDTFF9rP+LBu2ADFuS9Tl2jcuPDaV6rjktW5SglM8uAGX2wkXmChEacxX2hzZ6VlRKJviHt3Mc9I9haMPlflEZ+9'
    'UWNVjLyC1+/AE692RYNrYpJ8LQxlos7okDJdmAQ+fz0tiZGxLNBut+8F3cU6au5tvWBDHwCez4CHMmBEe08VcXCayjVh7x15VJC/'
    'Q2tWY9bRHYa3GIl+NX75bpEAv1+/J93wq3/02P/oaT141XIXvnowykp4horcUATmFw+D7VpTOMLLtkiq8YtPQSvLyr9PHT7VmHuY'
    'NhAuSirNV0+lsTtPh0XJETYl1jhAUS0hKcS3utvP4wPhvyp2p0LhdeS7KLPBLOzrmdT8ykeXX1j2aFeW87+/5s183YjRstxrUlGA'
    'oa9o87U9OUI0Seyluj5X6bYooTqdYlxSVidJDV8iZ9vUk5bm6vymU6wt8bxna22TIRulct5yCKabblrOSDCQfgdOOdcqrOvoBB/y'
    'gg2RSx2g3PCpIuee11Xt2z6lm1yQ9pPyg9CCxFr9vHG18WVjxgsZd7HkG9fzjREFwoGJ8O0cSE3cozRIdFCdzjKznEK3fVAun1HS'
    '1tl++NOCPyGiVe704JMkOEdNMiKxAckuQfLTnZYM23T70/Hb3imnVeZrNuBdIhX7k5xHvnJ+f3F+z5zf1HHn8XruPNBg1COEhu7r'
    '7tEp7rWQNk+UZfrk192jox5Mt7sHvS7fdqLThMqsCxpVigx6NVKYtUKNOsjjJFmREi3ahkXSTlam01SVhXb4PBgttVilianafm56'
    'vy7f/7Ud1eiOiMD6dDMREF1UOxADrGhTK1/U0OiXDRDWJ/h8bNJ/z2mq6BsFqMqhSeVgDlr0HM2NuWGQK3vJ1KsP2bF4UalLAPAf'
    'JYMVbEU75lIQnWlaTSCn5ZdF+evuAc7EH344OA1mm9AU9K7zWExboAYj9lUDMnQAs743So9Qhkahy3RYmnMYWThuWicPFfWFpg8h'
    '3mnVjKe0I74yybwZzBizinsEHhbaUjkb4B3Z6PdxIc4/6JYWuumpFf34o84QPBg05xdRaXPqJ59MGzalx5CufxRGredGziidOBkJ'
    '75891uU55sxVfrGeLzyezzSPQ8By+5j+wReP6Qd1dZh+VFekWYbsQKMUjcJCVPOqVDAIveBsoXsnIRsv2Yn2MOKw3pTklfqJUBTy'
    '1wXWqvwnxvYk1+xQLoMIJcW4lRFHoZiG6lInZtsUmvLatGlO/K78xJehEYfm+UATZBEZebq0sjGoliiLqZ3/o+5xd9+d/qk79wqx'
    'HR/dUsdkZtDCRvKYUq8SylmmyqxOW+YDcXBpL+mT8ZrsYzAV0LEEmwKYs7uowwk4VvCnabXT+ksutJLMvbY3G5HkKXODTxRPofwF'
    'cFpo0uGnwD+t9jjqPe5wDVlzK8SpJ6gTOJX3FmYcg1HIXZSTP1bHYfT6RStGxm0uUct5sXQuhOH0h/64+G97Icz/0WBgEiFfgjOm'
    'b9IB7eDFAs/Ct4sR1+oaVnqlKp27lV6ZSjrDD3EXJiROHBs7yR4lId+qiwn1gtKeGpHAZuOxRdw9y0c6yu02NWHvPdTg5WJYK1JT'
    '51b0+Bs6oCuoiyo9OwJ2/V26726/uAXl5Ouz8XqPZVadsolND9LNSm53uswnc05v5mSWG5FbKqMMq4iFkKNYet+QTbYtg7riW1nG'
    'sXsflw6ibBYPdMbAe0Y5teYVpAD20Xb/fjBcelvyg9m/YvDd5y9/WJH4i8Ct3JxVWw1bM3c+Pb9LJcFLKEJ/64okH52HZiWAuK30'
    'rzinh/uKpawrJWIpjNYFLUG0FrDs3SL2pkbeyDzIKtslZ8qz0pgV/1d1I7x2qEm67RrKVPlxnujcnnwQiUDnlG7MkOr5HXUybyZR'
    '2jmuVCLWTGVLNWuU08/JZuvY8UFwiGwbpXp/CHaPe8nvUYtXs0EYF9KYr1x96lBrKU5KuyMn9bqvHlf3VeKQ8SpJKaRiUK9gxAhI'
    'ziJYt6BWCUpKUFFAGFc1xMdWTvpZS0pSQfL+qso/PaAIdUO4ONrI2mNtTjs+08tUqtibgg5EFKXoQU17qqUERURiCGDCZPJB3EtO'
    'HqHM3kG6vIQB4OIySBdMZl+FemFp7ublZI/1pSaTPFa5BSSLB00CHB3VY5jSfHYTP9eZAG3VNlWU9J1nJWez/J9mVxfu0Q4zXEvo'
    'jkQUxPWp0a9O2XBBtlF1zI3LKtokwVSvAXeMwizdN4ZJtZlbXlh6NtM6JW+hp4j+edWJJKUdfAY/JDrpwBdJOeUAR6zMFP356yG+'
    'ubJHP/2LjPOCTCmzZtLUlGOZ5DfRYG1XlvVhkvnwmDvmTLE6YEpbh9zQenK0t3uqjFzqRpYBL37voni5eZGvaVl1vtsFpzUVVcU0'
    'tos7vl0FyRwYdyt3ave0rDgjXs8kVruoJTiwZPvJPQmP6dM1OEnzNbV+/52nfqfhvttBoGTRRKj4dJOz34Ghsvs7NyC5+OA51fdX'
    'O3ee0HXVKa6rNvedGPmWL9Pm7Ak4JYD5N6qWGSZF9eKDvSkHL0LkmfhhshGr62IZcizBW3TQ0Lvlu2VubjHn4SeU18e/e9t4Si+L'
    '+jebv4tyypDk42XvNPRi0EGEqahEv6tThwstKMUj7U9mnctC5xbNL4KcoTQqAyIkKVXcR5ApLViqX0reCoAa7kjX3mBJq+mo3dGk'
    'Au45iYYJWda5DFSewXpqA2VuusSVItwu9RnMLGq4LJ29Zwz+p+ilO0556fu+hnbgcvdTrNv23WByEE4PjQ/ObXOej0gfoqMePXgH'
    'KRUKE7/oA4aUF5m+O6sAl8fzFstLwMSo15OD0OGU5pc6iZQ2Tdj76oMDYmhATuwTr61dhxSuNwWqpWmnvl4fnaGk8NKhmf6FxfGh'
    '8c4qWtzPmpBfy83JiS6GDR5kKqgitSzOGvzH9qPJMcmpwwzykxV1vPGru2t02iU9a3pSnzlV1E3ifmoUU0lC+vUhDIqR6PU71CEF'
    'Si+1HfnkfunoxKLFlNqOUfsXlCO3HZ01AUPHcuqefjjuuYlP+AwbE6Kbm6QRBZiPVIWEik5BrnPKAtKiWn5YxDndjGVPenjVJLTi'
    'wXrqQIgmYnu0w1s3D9Dz4xiXORzelBLKwUPICmzO1v7zQf2cbSobgMdhuAozmQZi1BimbH+fBvUwH3eByIpCprb+1qCWoqwyYBz6'
    'rq0aPulC9FabX/2TbGxET7532JlR/bO5IIGZOLnFcJIIMjdcpFLWix3XTVL2jfwfiUoE0JgoQ44LU+4VyjtBoQuxbmmdyV/3E4qa'
    'CJZYDlLpqe2x0oB04ku+zs3/pL64uSHs6TTF4HUTRoOi/4mYZOzunGWgZu7m/HGebf7BHHR+FQbrVAhvAUH241+PThuvMoZTRt9k'
    'DP1zKZcKL1kifgXh8qE6Wgumq9F0vRMEzZN7DL6xvT8s6SqJsYf8vq9PtIr3bhelj/9Qw+YW32yZy+j4uXd0uPOOX6kXuPf01Caq'
    'NKeqrxfkzhcdHO6S4dJN/P/JWGns7XePSvTn1dXbrJM4jnU4F/Vu+1qfC1elgSn65ZcVhn2Pvv40yKYDM4Gtic/qsL++40D/K3AD'
    '2K7RQd0YDs9zmxK5ZtKgz32nUSdT2gq0sWymqqnogRVYdREWdC32AThaA+Ie4aoDW6ETI0ZjCLVLnafNzx5mNlltZdZqX3DU6t4T'
    'Kax/6LtAHd+lb6AJskrYnBJA3PGHA9HDGFWhkuUI7iQbVinxlHw+ww0G4T0CtaobkdLsWFQ+Oe0dmcOvCAKCTEEASVKg1US3/wYt'
    'KAkc96yYomSw2+Ibb2LcbLb/4Sh9c9zdYTuigRimFZO6ZPxgqKwqBi2pOw8JhnyS+4YcmDrijZOKsij7pzrgRJFpkctT+Ugar4i9'
    '4798hchhkaFKJICd03EWuFPLwuaXaEBnYMSxsSmnrTDyD71pw7Ay/U1SoYJaJJoV9DrM7rZ/6+2+fQeu2dvp/qF9+XAvpZNFKKpf'
    'sGVdtyCcPBWYbtCmqqzHwOe1nEXKcW62f9i12m9hlDnh1+ZibD/Tqtxb7QWPTsAMqEpMExMY2+gVdDBLMp3GuDquKmn8N5zCHofF'
    'XR5so5Sy606xROUc2fIoxC9SC1TbbG/xAfV/gFZxTF2f4n4mwa24oUYW0pyuLNCtJ2bchtmfrcnKAAFsMoOdVeY3tWwfcqiCI/nY'
    'lqISpMps7y8NmNZ/7+TDfs9LKirBt3ZXUiX1hpvf0qG02Ozf9ezRAsAUeOBuBynNV9OTMZgu8YxBOxtX2QWiYTf4zhl1p3UoDzi9'
    'Ezir7azEwx1Zj27qlqNZSPNt7gzdVpeu66WCQ9fwMdbuGzLbU8P21Vd3bVnCN2dHzQpasb/MaFRhLVk2q6rI14ZaimJ0MfbArwVq'
    'gJV0VDHEewM/d7WCVghSBfnaWT7d2lBQS0duWUW2TcW17OQWb+6vmeW/OcTVFHZ9yZe4cq4BNcmWWEO5RzYJusCXUsRSDC1Nv6pG'
    '24LqektFFJglqC5Zl2Vnv8rzwONZ5IpJiXY9uUtR5J+GYDqNEiNHSc75dlVLQWEBTQcdTS9egTqKDEl0pON8ZYye/Y6mDL7iTk91'
    'R5OBeWt4kZ32xqbcKe04k+6IrrIsCU9M2c6EgUfCAb+czr0FD16EnBt0piduKulPuBaLmCVan7iZMc0rtaUwRdQAsQo4KNr0lxah'
    '4H1C3B9dNtX0ldIuKMeWcRXRu9IaHeuSmd1kRagyiePqxCZiUm3Hk2LC3QcNV0krXXPqZ64PhbUwoEWbZi+vx2OyzeaLaROHcVre'
    '5lZWlKFtauBfhKT6FIDx7HLu51nlfRW7LJX5SazC1MGQ41Ma+FRS+NH3Pop3xHoXCqYhoyEdKcjP5+U6N6Cb0kNQZS0EuU6PopG1'
    '07xYkSbkIWHrFy0tZ2nPZrNcZopVTXc3GNFMHXMicE+jhoMIip5Ds3DjDiVDoFKJ56BtKIqgHPqR2s4H5VSEQsm3cSONzKSYy7WF'
    'JR/08UwYz3wB+e0x3ZK1t3u0opcYyspmdff4hGS8cvvmszYcO8ELepZSMlhtA6pZu3QiPEMzyQOLhH40llBkETgUPN5jfMnsJ9c6'
    'fFIvXSoodeN00NgTxMe+TXsf4ejgpXLSlHFNZLtsXkk+f8saIRZbrtmQUYRV+5xO4enqG2F7DZlL8rHkf3Wl76DfDW2prV4Y7RO6'
    'QpybfjLa4AfGH93xR78XlGM6x68n7S3gZINLQhbAsZ0n7U1KoFlKZGdzuhGlVQZ98hSGlp000Sd4ViiFIcK9jvVyXJHOROOtJbh4'
    'ahG5Ef1AXvem7CsN+1OdKjwebZbjyo1vBaGaHZWPB3n7Lg7/hvSBlpNH0OFJ92PvMYToSFs1qH7TkEKOT9PT3f0etPGT3snJLlz3'
    'O90jMgl+C2A+SxGYGK3puokIT6Df8Rm7gu6pxUVDiFdEvM1InXqQS7o5gqcddZe0iuU0JpSbHG6IK7pkVEVVIySolOth2YJDskt2'
    'QSfacWhPxNq2JzoFl+o6yPGGEG6m+gpeGy6K/Y2PF7FVhS7tLfTBcjJHlSFRaEmG1jovwWcuF7tPVKiLGr5uKrkqfPXY2HW0WhzQ'
    '01/+lYfQGb81dsMuDjL2NUmEDXmTfIOhX4yHIcY80a1IK95aXcZJZMTbxuqSOsHRwD9G3cgVKT1OBUoEt6Nz0PuUimb9zdYvT4iz'
    'M3tEKnn6l87c853mxBcJA8QYK+GKCvucqoZG3lI4CFc+1UvqqxSo+llrGw3r0Kqj460VX7UO6hgSVyqeTUpXAw+0Goqt3YqMRyJZ'
    'vdjJ13QTkRakF8fa/fddepPCNjFNzXp2PAhKW9MyJ8+8eyK7iW0By86FRyoPPVvgOk2iGH2SM+3I+Q5GZXl/6ENZpVWpVbaKx9RC'
    'f0ZkvVf7L3Zb/GQEEk1i6pkQcaBEk62Qnd8X2QpbVm21lBFcc+6Qg1KgyB3qMoTOPYaLh+nnEbTjUEHd1GYLOv3w+uCYtUzhbzZt'
    'BZT4TZa5Jq/bN1jnVg3mX7PR1aa00YumDt+hGXA73mtNMK6CbEYRYEbXPXAuAKQdA+GKjXDckO+iarghkD+11d0jTSuFwd87piZ7'
    'pK0Q+ER4KelOKgzEK5aWOsK0YhUFzq61wX8DVokLbg=='
)

In [ ]:
import os
import sys
import io
import gc
import json
import time
import zlib
import base64
import contextlib

import torch

# Folders searched for inputs. On Kaggle this is /kaggle/input (the env var is only for local tests).
INPUT_ROOTS = os.environ.get("PCR_INPUT_ROOTS", "/kaggle/input").split(":")


def _walk_all():
    for root in INPUT_ROOTS:
        if not os.path.isdir(root):
            continue
        for dir_path, dir_names, file_names in os.walk(root):
            yield dir_path, file_names


def _prefer_final(paths):
    paths = sorted(paths)
    for p in paths:
        if "final" in p.lower():
            return p
    if len(paths) > 0:
        return paths[0]
    return None


def find_assets():
    # Locate the data bundle, the two Phase-1 model folders and the two model code folders.
    # Model 1 and Model 4 both have a file called legal_gqa_model.py, so the folder that also holds moe.py is Model 4.
    assets = {"bundle": None, "m1_dir": None, "m1_py": None, "m4_dir": None, "m4_py": None}
    m1_dirs = []
    m4_dirs = []
    for dir_path, file_names in _walk_all():
        low = dir_path.lower()
        if "tokenized_pcr.pkl" in file_names and assets["bundle"] is None:
            assets["bundle"] = os.path.join(dir_path, "tokenized_pcr.pkl")
        if "config.json" in file_names and "checkpoint" not in low:
            model_type = ""
            try:
                with open(os.path.join(dir_path, "config.json")) as f:
                    model_type = str(json.load(f).get("model_type", ""))
            except Exception:
                model_type = ""
            if model_type == "legal_gqa_mha_moe" or "gqamoe" in low:
                m4_dirs.append(dir_path)
            elif "legal-gqa" in low or "legal_gqa" in model_type:
                m1_dirs.append(dir_path)
        if "model.py" in file_names and "moe.py" in file_names and "legal_gqa_model.py" in file_names:
            if assets["m4_py"] is None:
                assets["m4_py"] = os.path.join(dir_path, "model.py")
        elif "legal_gqa_model.py" in file_names and "moe.py" not in file_names:
            if assets["m1_py"] is None:
                assets["m1_py"] = os.path.join(dir_path, "legal_gqa_model.py")
    assets["m1_dir"] = _prefer_final(m1_dirs)
    assets["m4_dir"] = _prefer_final(m4_dirs)
    return assets


def find_run_dirs():
    # Finished PCR runs: a folder holding run_summary.json and adapter_best.pt.
    found = {}
    for dir_path, file_names in _walk_all():
        if "run_summary.json" not in file_names or "adapter_best.pt" not in file_names:
            continue
        try:
            with open(os.path.join(dir_path, "run_summary.json")) as f:
                summ = json.load(f)
        except Exception:
            continue
        if "pooling" not in summ or "adapter" not in summ or "model" not in summ:
            continue
        model = "m4" if "Model 4" in str(summ["model"]) else "m1"
        adapter = "lorma" if str(summ["adapter"]).lower() == "lorma" else "lora"
        key = model + "_" + adapter
        if summ.get("kd"):
            key = key + "_kd"
        if key not in found:
            found[key] = []
        found[key].append(dir_path)
    runs = {}
    for key in sorted(found):
        paths = sorted(found[key])
        runs[key] = paths[0]
        if len(paths) > 1:
            print("note: several runs match %s, using %s (others: %s)" % (key, paths[0], paths[1:]))
    return runs


@contextlib.contextmanager
def _env(values):
    old = {}
    for k in values:
        old[k] = os.environ.get(k)
        os.environ[k] = str(values[k])
    try:
        yield
    finally:
        for k in old:
            if old[k] is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = old[k]


def _apply_settings(ns, summ):
    # The run replays with exactly the settings it was trained with (read from its run_summary.json).
    ns["RUN_TRAIN"] = False
    ns["RESUME"] = False
    ns["RUN_TEST"] = False
    ns["POOLING"] = summ["pooling"]
    ns["DOC_VIEW"] = summ["doc_view"]
    ns["HEAD_CHUNKS"] = int(summ["head_chunks"])
    ns["TAIL_CHUNKS"] = int(summ["tail_chunks"])
    ns["CHUNK_LEN"] = int(summ["chunk_len"])
    ns["CONTENT_LEN"] = ns["CHUNK_LEN"] - 1
    ns["USE_PROJ"] = int(summ.get("projection_params", 0)) > 0
    prefix = "LORMA_" if str(summ["adapter"]).lower() == "lorma" else "LORA_"
    settings = summ.get("adapter_settings", {})
    for k in settings:
        ns[prefix + k.upper()] = settings[k]


def _load_cells(key):
    blob = _SOURCES_B64[key]
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode())


def _free(ns):
    ns.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    sys.modules.pop("legal_moe_model", None)


def embed_run(key, run_dir, splits, assets):
    # Rebuild one finished run (frozen Phase-1 model + its adapter + projection) and embed the given splits.
    with open(os.path.join(run_dir, "run_summary.json")) as f:
        summ = json.load(f)
    src_key = key[:-3] if key.endswith("_kd") else key
    model = src_key.split("_")[0]
    if assets[model + "_dir"] is None or assets[model + "_py"] is None:
        raise FileNotFoundError("Phase-1 %s folder or code not found among the inputs (needed for %s)" % (model, key))
    cells = _load_cells(src_key)
    env = {"PCR_OUT_DIR": run_dir, "PCR_MODEL_DIR": assets[model + "_dir"], "PCR_MODEL_PY": assets[model + "_py"]}
    if assets["bundle"] is not None:
        env["PCR_DATA_DIR"] = os.path.dirname(assets["bundle"])
    ns = {"__name__": "__main__"}
    buf = io.StringIO()
    t0 = time.time()
    try:
        with _env(env):
            with contextlib.redirect_stdout(buf):
                for i, src in enumerate(cells):
                    exec(compile(src, "<%s cell %d>" % (key, i), "exec"), ns)
                    if i == 0:
                        _apply_settings(ns, summ)
                if ns.get("BEST_STATE") is None:
                    raise RuntimeError("adapter_best.pt was not loaded from " + run_dir)
                if int(ns["N_ADAPTER"]) != int(summ["adapter_params"]):
                    raise RuntimeError("adapter size %d does not match the run summary (%d) for %s" % (
                        int(ns["N_ADAPTER"]), int(summ["adapter_params"]), key))
    except Exception:
        print(buf.getvalue()[-3000:])
        raise
    shown = []
    for line in buf.getvalue().split("\n"):
        for word in ["document view used", "model:", "trainable", "adapter target", "loaded saved adapter"]:
            if word in line:
                shown.append(line)
                break
    for line in shown:
        print("   [%s] %s" % (key, line))
    t_load = time.time() - t0
    result = {}
    t1 = time.time()
    for split in splits:
        result[split] = ns["embed_split"](split)
    info = {"key": key, "base_params": int(ns["N_BASE"]), "adapter_params": int(ns["N_ADAPTER"]),
            "embedding_dim": int(ns["D_MODEL"]), "pooling": ns["POOLING"], "doc_view": ns["DOC_VIEW"],
            "load_seconds": round(t_load, 1), "embed_seconds": round(time.time() - t1, 1)}
    _free(ns)
    return result, info


def make_scoring_ns(out_dir):
    # Scoring functions (cosine, CSLS, dual softmax, BM25, F1) taken from the same code the runs used. No model is loaded.
    cells = _load_cells("m1_lora")
    order = [0, 1, 2, 5]
    env = {"PCR_OUT_DIR": out_dir, "PCR_MODEL_DIR": "unused", "PCR_MODEL_PY": "unused"}
    assets = find_assets()
    if assets["bundle"] is not None:
        env["PCR_DATA_DIR"] = os.path.dirname(assets["bundle"])
    ns = {"__name__": "__main__"}
    buf = io.StringIO()
    try:
        with _env(env):
            with contextlib.redirect_stdout(buf):
                for i in order:
                    exec(compile(cells[i], "<scoring cell %d>" % i, "exec"), ns)
    except Exception:
        print(buf.getvalue()[-3000:])
        raise
    return ns

In [ ]:
ASSETS = find_assets()
RUNS = find_run_dirs()
print("inputs found:", {k: ASSETS[k] for k in ASSETS})
print("finished runs found:", RUNS)
for key in TEACHER_KEYS:
    if key not in RUNS:
        raise FileNotFoundError("teacher run %s not found. Attach its output (adapter_best.pt + run_summary.json)." % key)

TEACH_CACHE = os.path.join(OUT_DIR, "teacher_embeddings.pt")
RAW = None
if os.path.exists(TEACH_CACHE):
    RAW = torch.load(TEACH_CACHE, map_location="cpu", weights_only=False)
else:
    cached = find_own_file("teacher_embeddings.pt")
    if cached is not None:
        RAW = torch.load(cached, map_location="cpu", weights_only=False)
        shutil.copy(cached, TEACH_CACHE)
if RAW is not None:
    ok = True
    for key in TEACHER_KEYS:
        if key not in RAW:
            ok = False
        else:
            for split in KD_SPLITS:
                if split not in RAW[key]:
                    ok = False
    if not ok:
        RAW = None
if RAW is not None:
    print("teacher embeddings loaded from the saved file")
else:
    RAW = {"_info": {}}
    for key in TEACHER_KEYS:
        t0 = time.time()
        result, info = embed_run(key, RUNS[key], KD_SPLITS, ASSETS)
        RAW[key] = result
        RAW["_info"][key] = info
        print("teacher %s embedded: %s | %.0fs" % (key, info, time.time() - t0))
    torch.save(RAW, TEACH_CACHE)
    print("saved", TEACH_CACHE)

# lookup tables: id -> row of the teacher embedding
TEACH = {}
for key in TEACHER_KEYS:
    TEACH[key] = {}
    for split in KD_SPLITS:
        t_qids, t_pids, t_Qe, t_Pe = RAW[key][split]
        if list(t_qids) != list(B["query_ids"][split]) or list(t_pids) != list(B["pool_ids"][split]):
            raise RuntimeError("teacher %s was embedded with a different %s query/pool order than this bundle" % (key, split))
        qrow = {}
        for i, q in enumerate(t_qids):
            qrow[q] = i
        prow = {}
        for i, p in enumerate(t_pids):
            prow[p] = i
        TEACH[key][split] = {"Qe": t_Qe.float(), "Pe": t_Pe.float(), "qrow": qrow, "prow": prow}

# how good are the teachers on dev (plain cosine)? this is the number the student has to approach
TEACH_DEV_F1 = {}
if "dev" in B["splits"]:
    S_sum = None
    for key in TEACHER_KEYS:
        t_qids, t_pids, t_Qe, t_Pe = RAW[key]["dev"]
        p, r, f1 = plain_f1(t_Qe, t_Pe, t_qids, t_pids, GOLD["dev"])
        TEACH_DEV_F1[key] = f1
        print("teacher %-9s dev Micro-F1@%d = %.4f (P %.4f, R %.4f) | base params %.1fM, embedding dim %d" % (
            key, TOP_K, f1, p, r, RAW["_info"][key]["base_params"] / 1e6, RAW["_info"][key]["embedding_dim"]))
        S_k = make_scores(t_Qe, t_Pe, t_qids, t_pids, False)
        if S_sum is None:
            S_sum = S_k
        else:
            S_sum = S_sum + S_k
    S_avg = S_sum / len(TEACHER_KEYS)
    lists = top_ids(S_avg, t_pids, TOP_K)
    p, r, f1 = micro_prf(lists, t_qids, GOLD["dev"], TOP_K)
    TEACH_DEV_F1["average"] = f1
    print("teacher average (the distillation target)  dev Micro-F1@%d = %.4f (P %.4f, R %.4f)" % (TOP_K, f1, p, r))

## 7. Zero-shot baseline on dev (frozen Phase-1 model, no adapter)
This is the "how well do Phase-1 representations transfer" row for the report, next to token-BM25.

In [ ]:
ZERO_SHOT = {}
if "dev" in B["splits"]:
    dev_gold = GOLD["dev"]
    t0 = time.time()
    saved_pooling = POOLING
    for pooling in ["mean", "last"]:
        POOLING = pooling
        qids, pids, Qe, Pe = embed_split("dev")
        p, r, f1 = plain_f1(Qe, Pe, qids, pids, dev_gold)
        S_center = make_scores(Qe, Pe, qids, pids, True)
        lists = top_ids(S_center, pids, TOP_K)
        pc, rc, f1c = micro_prf(lists, qids, dev_gold, TOP_K)
        ZERO_SHOT["frozen_" + pooling] = f1
        ZERO_SHOT["frozen_" + pooling + "_centered"] = f1c
        print("frozen Phase-1, %-4s pooling: Micro-F1@%d = %.4f (P %.4f, R %.4f) | centered: %.4f" % (pooling, TOP_K, f1, p, r, f1c))
    POOLING = saved_pooling
    qids = list(B["query_ids"]["dev"])
    pids = list(B["pool_ids"]["dev"])
    dev_bm25 = bm25_scores("dev", qids, pids)
    lists = top_ids(dev_bm25, pids, TOP_K)
    p, r, f1 = micro_prf(lists, qids, dev_gold, TOP_K)
    ZERO_SHOT["token_bm25_headtail"] = f1
    print("token-BM25 (head+tail %d tokens): Micro-F1@%d = %.4f (P %.4f, R %.4f)" % (BM25_VIEW_TOKENS, TOP_K, f1, p, r))
    print("word-level BM25 from the data-prep notebook (head+tail 1024 words):", view_results.get("headtail"))
    print("zero-shot dev evaluation took %.0fs" % (time.time() - t0))
else:
    print("No dev split: skipping the zero-shot baseline.")

## 8. Hand-written LoRA

LoRA: `y = W0 x + (alpha / r) * B A x`. `W0` is frozen, `B` starts at zero, so training starts exactly at the Phase-1 weights.

In [ ]:
def hand_dropout(x, p, training):
    # dropout written by hand: zero some entries, scale the rest
    if (not training) or p == 0.0:
        return x
    keep_prob = 1.0 - p
    rand = torch.rand_like(x)
    mask = (rand < keep_prob).to(x.dtype)
    return x * mask / keep_prob


def hand_uniform_(tensor, bound):
    # fill a tensor with numbers drawn uniformly from [-bound, bound]
    with torch.no_grad():
        tensor.copy_(torch.rand_like(tensor) * 2.0 * bound - bound)


class LoRALinear(nn.Module):
    # y = W0 x + (alpha / r) * B (A x)       W0 frozen, only A and B are trained
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out, d_in = base.weight.shape
        self.d_in = d_in
        self.r = r
        self.scale = alpha / r
        self.p_drop = dropout
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, d_in, device=dev, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r, device=dev, dtype=torch.float32))
        self.reset_parameters()

    def reset_parameters(self):
        # A: uniform in [-1/sqrt(d_in), 1/sqrt(d_in)], B: zeros, so the update starts at exactly 0
        bound = 1.0 / math.sqrt(self.d_in)
        hand_uniform_(self.lora_A, bound)
        with torch.no_grad():
            self.lora_B.zero_()

    def forward(self, x):
        y = self.base(x)
        x32 = x.float()
        x32 = hand_dropout(x32, self.p_drop, self.training)
        low = x32 @ self.lora_A.t()        # (.., r)
        delta = low @ self.lora_B.t()      # (.., d_out)
        delta = delta * self.scale
        return y + delta.to(y.dtype)


def set_submodule(root, dotted_name, new_module):
    parts = dotted_name.split(".")
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


# Targets: every linear layer inside the blocks. Not the embedding, not lm_head.
ADAPTER_TARGETS = []
for name, module in LM.named_modules():
    if isinstance(module, nn.Linear) and name.startswith("blocks."):
        ADAPTER_TARGETS.append(name)
if len(ADAPTER_TARGETS) == 0:
    raise RuntimeError("No linear layers found under blocks. Check the Model 1 module names.")
print("LoRA target layers:", len(ADAPTER_TARGETS), "| first few:", ADAPTER_TARGETS[:5])

with torch.no_grad():
    check_ids = torch.tensor([[5, 9, 12, 40, 77, SEP_ID]], device=DEVICE)
    before = backbone_hidden(check_ids).float().clone()

ADAPTER_MODULES = []
for name in ADAPTER_TARGETS:
    wrapped = LoRALinear(LM.get_submodule(name), LORA_R, LORA_ALPHA, LORA_DROPOUT)
    set_submodule(LM, name, wrapped)
    ADAPTER_MODULES.append(wrapped)

with torch.no_grad():
    after = backbone_hidden(check_ids).float()
identity_diff = float((before - after).abs().max())
print("identity check (max abs diff at init, should be 0):", identity_diff)
assert identity_diff < 1e-4, "LoRA changed the Phase-1 output at init"

ADAPTER_PARAMS = []
for m in ADAPTER_MODULES:
    ADAPTER_PARAMS.append(m.lora_A)
    ADAPTER_PARAMS.append(m.lora_B)
N_ADAPTER = sum(p.numel() for p in ADAPTER_PARAMS)
print("trainable LoRA params: %s (%.2f%% of the model)" % (f"{N_ADAPTER:,}", 100.0 * N_ADAPTER / N_BASE))


def adapter_state():
    state = {}
    for name, m in zip(ADAPTER_TARGETS, ADAPTER_MODULES):
        state[name] = (m.lora_A.detach().cpu().clone(), m.lora_B.detach().cpu().clone())
    return state


def load_adapter_state(state):
    for name, m in zip(ADAPTER_TARGETS, ADAPTER_MODULES):
        a, b = state[name]
        with torch.no_grad():
            m.lora_A.copy_(a.to(m.lora_A.device))
            m.lora_B.copy_(b.to(m.lora_B.device))

## 9. Training: multi-positive contrastive loss
Per step: a batch of queries, a few sampled positives and BM25 hard negatives each, plus every other document in the step as an in-batch negative. A document that is gold for a query is never a negative for it (and neither is the query itself). The loss for a positive `p` of query `q` is `-log( e^{s(q,p)/t} / ( e^{s(q,p)/t} + sum over negatives e^{s(q,n)/t} ) )`, averaged over the sampled positives, so positives do not compete with each other.

In [ ]:
def set_train_mode():
    LM.eval()                      # Phase-1 dropout stays off
    for m in ADAPTER_MODULES:
        m.train()                  # adapter dropout on


def set_eval_mode():
    LM.eval()
    for m in ADAPTER_MODULES:
        m.eval()


# training items: (split, query id)
TRAIN_SPLITS = ["train"]
if TRAIN_ON_DEV and "dev" in B["splits"]:
    TRAIN_SPLITS.append("dev")
TRAIN_ITEMS = []
for split in TRAIN_SPLITS:
    for qid in B["query_ids"][split]:
        if len(GOLD[split][qid]) > 0:
            TRAIN_ITEMS.append((split, qid))
if SMOKE_TEST:
    TRAIN_ITEMS = TRAIN_ITEMS[:SMOKE_TRAIN_QUERIES]
print("training queries:", len(TRAIN_ITEMS), "| splits:", TRAIN_SPLITS)

HARD_NEG = B.get("hard_neg", {})


def sample_query(item):
    split, qid = item
    gold = GOLD[split][qid]
    pos = random.sample(gold, min(POS_PER_QUERY, len(gold)))
    lo = HARD_NEG_RANGE[0]
    hi = HARD_NEG_RANGE[1]
    pool_hard = []
    if split in HARD_NEG and qid in HARD_NEG[split]:
        pool_hard = HARD_NEG[split][qid][lo:hi]
    negs = []
    if len(pool_hard) > 0:
        negs = random.sample(pool_hard, min(HARD_NEG_PER_QUERY, len(pool_hard)))
    gold_set = GOLDSET[(split, qid)]
    tries = 0
    pool_ids = B["pool_ids"][split]
    while len(negs) < HARD_NEG_PER_QUERY and tries < 50:
        tries += 1
        c = random.choice(pool_ids)
        if c in gold_set or c == qid or c in negs:
            continue
        negs.append(c)
    return pos, negs


KD_LOG = {"con": 0.0, "kd": 0.0, "n": 0}


def kd_term(q_specs, cand_specs, Qv, Cv):
    # KL( teacher softmax || student softmax ) over the candidates of this step, per query, averaged over queries.
    n_q = len(q_specs)
    n_c = len(cand_specs)
    T = None
    for key in TEACHER_KEYS:
        tq_rows = []
        for spec in q_specs:
            tq_rows.append(TEACH[key][spec[0]]["Qe"][TEACH[key][spec[0]]["qrow"][spec[2]]])
        tc_rows = []
        for spec in cand_specs:
            tc_rows.append(TEACH[key][spec[0]]["Pe"][TEACH[key][spec[0]]["prow"][spec[2]]])
        Tq = torch.stack(tq_rows)
        Tc = torch.stack(tc_rows)
        sc = Tq @ Tc.t()
        if T is None:
            T = sc
        else:
            T = T + sc
    T = (T / len(TEACHER_KEYS)).to(Qv.device)
    valid = torch.ones(n_q, n_c, dtype=torch.bool)
    for i, qs in enumerate(q_specs):
        for j, cs in enumerate(cand_specs):
            if cs[2] == qs[2]:
                valid[i, j] = False        # a query is in its own pool: never a candidate for itself
    valid = valid.to(Qv.device)
    student = (Qv @ Cv.t()) / KD_TEMPERATURE
    teacher = T / KD_TEMPERATURE
    log_s = torch.log_softmax(student.masked_fill(~valid, -1e9), dim=1)
    log_t = torch.log_softmax(teacher.masked_fill(~valid, -1e9), dim=1)
    p_t = log_t.exp()
    kl = (p_t * (log_t - log_s)).sum(1)
    return kl.mean()


def step_loss(items):
    q_specs = []
    cand_specs = []
    cand_index = {}
    positives = []
    for item in items:
        split, qid = item
        pos, negs = sample_query(item)
        q_specs.append((split, "q", qid))
        positives.append((split, pos))
        for c in pos + negs:
            spec = (split, "c", c)
            if spec not in cand_index:
                cand_index[spec] = len(cand_specs)
                cand_specs.append(spec)
    all_specs = q_specs + cand_specs
    E = embed_docs(all_specs, True)
    Qv = E[:len(q_specs)]
    Cv = E[len(q_specs):]
    sims = (Qv @ Cv.t()) / TEMPERATURE
    n_q = len(items)
    n_c = len(cand_specs)
    neg_mask = torch.ones(n_q, n_c, dtype=torch.bool)
    pos_mask = torch.zeros(n_q, n_c, dtype=torch.bool)
    for i, item in enumerate(items):
        split, qid = item
        gold_set = GOLDSET[(split, qid)]
        for j, spec in enumerate(cand_specs):
            c_id = spec[2]
            if c_id in gold_set or c_id == qid:
                neg_mask[i, j] = False
        for c in positives[i][1]:
            pos_mask[i, cand_index[(split, "c", c)]] = True
    neg_mask = neg_mask.to(sims.device)
    pos_mask = pos_mask.to(sims.device)
    neg_lse = torch.logsumexp(sims.masked_fill(~neg_mask, -1e9), dim=1, keepdim=True)
    terms = F.softplus(neg_lse - sims)
    per_query = (terms * pos_mask.float()).sum(1) / pos_mask.float().sum(1).clamp(min=1.0)
    con_loss = per_query.mean()
    kd_loss = kd_term(q_specs, cand_specs, Qv, Cv)
    KD_LOG["con"] += float(con_loss.detach())
    KD_LOG["kd"] += float(kd_loss.detach())
    KD_LOG["n"] += 1
    return con_loss + KD_WEIGHT * kd_loss


TRAIN_PARAMS = list(ADAPTER_PARAMS)
PROJ_PARAMS = []
if PROJ_MODULE is not None:
    PROJ_PARAMS = list(PROJ_MODULE.parameters())

CKPT_PATH = os.path.join(OUT_DIR, "ckpt_pcr.pt")
BEST_PATH = os.path.join(OUT_DIR, "adapter_best.pt")
STOPPED_EARLY = False
TRAIN_SECONDS = 0.0
HISTORY = []
BEST_F1 = -1.0
BEST_EPOCH = -1
BEST_STATE = None


def current_state_dict():
    proj_state = None
    if PROJ_MODULE is not None:
        proj_state = {}
        for k, v in PROJ_MODULE.state_dict().items():
            proj_state[k] = v.detach().cpu().clone()
    return {"adapter": adapter_state(), "proj": proj_state}


def load_state_dict_all(blob):
    load_adapter_state(blob["adapter"])
    if PROJ_MODULE is not None and blob["proj"] is not None:
        PROJ_MODULE.load_state_dict(blob["proj"])


def quick_dev_f1():
    set_eval_mode()
    qids, pids, Qe, Pe = embed_split("dev")
    p, r, f1 = plain_f1(Qe, Pe, qids, pids, GOLD["dev"])
    set_train_mode()
    return p, r, f1


if RUN_TRAIN and len(TRAIN_ITEMS) > 0:
    steps_per_epoch = int(math.ceil(len(TRAIN_ITEMS) / QUERIES_PER_STEP))
    total_steps = EPOCHS * steps_per_epoch
    warm_steps = max(1, int(WARMUP_FRAC * total_steps))
    print("steps per epoch:", steps_per_epoch, "| total steps:", total_steps)

    groups = [{"params": TRAIN_PARAMS, "lr": LR}]
    if len(PROJ_PARAMS) > 0:
        groups.append({"params": PROJ_PARAMS, "lr": PROJ_LR})
    optimizer = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    base_lrs = []
    for g in optimizer.param_groups:
        base_lrs.append(g["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    def lr_scale(step):
        if step < warm_steps:
            return (step + 1) / warm_steps
        progress = (step - warm_steps) / max(1, total_steps - warm_steps)
        return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

    state = {"epoch": 0, "pos": 0, "step": 0, "seconds": 0.0, "perm": None}

    if RESUME:
        found = None
        if os.path.exists(CKPT_PATH):
            found = CKPT_PATH
        else:
            found = find_own_file("ckpt_pcr.pt")
        if found is not None:
            blob = torch.load(found, map_location="cpu", weights_only=False)
            load_state_dict_all(blob["model"])
            optimizer.load_state_dict(blob["opt"])
            scaler.load_state_dict(blob["scaler"])
            state = blob["state"]
            HISTORY = blob["history"]
            BEST_F1 = blob["best_f1"]
            BEST_EPOCH = blob["best_epoch"]
            BEST_STATE = blob["best_state"]
            if found != CKPT_PATH:
                shutil.copy(found, CKPT_PATH)
            print("resumed from", found, "| epoch", state["epoch"], "| step", state["step"])

    def save_ckpt():
        blob = {"model": current_state_dict(), "opt": optimizer.state_dict(), "scaler": scaler.state_dict(),
                "state": state, "history": HISTORY, "best_f1": BEST_F1, "best_epoch": BEST_EPOCH,
                "best_state": BEST_STATE}
        torch.save(blob, CKPT_PATH + ".tmp")
        os.replace(CKPT_PATH + ".tmp", CKPT_PATH)

    seconds_before = state["seconds"]
    t_run = time.time()
    t_last_log = time.time()
    run_loss = 0.0
    run_n = 0
    set_train_mode()

    for epoch in range(state["epoch"], EPOCHS):
        if state["perm"] is None:
            perm = list(range(len(TRAIN_ITEMS)))
            random.shuffle(perm)
            state["perm"] = perm
            state["pos"] = 0
        perm = state["perm"]
        pos = state["pos"]
        while pos < len(perm):
            batch_idx = perm[pos:pos + QUERIES_PER_STEP]
            items = []
            for i in batch_idx:
                items.append(TRAIN_ITEMS[i])
            scale = lr_scale(state["step"])
            for g, base_lr in zip(optimizer.param_groups, base_lrs):
                g["lr"] = base_lr * scale
            loss = step_loss(items)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(TRAIN_PARAMS + PROJ_PARAMS, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            pos += len(batch_idx)
            state["pos"] = pos
            state["step"] += 1
            run_loss += float(loss.detach())
            run_n += 1

            if state["step"] % LOG_EVERY_STEPS == 0:
                elapsed = time.time() - t_last_log
                per_step = elapsed / LOG_EVERY_STEPS
                left = total_steps - state["step"]
                print("epoch %d | step %d/%d | loss %.4f | lr %.2e | %.1f s/step | eta %.0f min" % (
                    epoch, state["step"], total_steps, run_loss / max(run_n, 1), LR * scale,
                    per_step, left * per_step / 60.0))
                print("   contrastive %.4f | distillation %.4f" % (KD_LOG["con"] / max(KD_LOG["n"], 1), KD_LOG["kd"] / max(KD_LOG["n"], 1)))
                KD_LOG["con"] = 0.0
                KD_LOG["kd"] = 0.0
                KD_LOG["n"] = 0
                run_loss = 0.0
                run_n = 0
                t_last_log = time.time()

            state["seconds"] = seconds_before + (time.time() - t_run)
            if state["step"] % SAVE_EVERY_STEPS == 0:
                save_ckpt()
            if time.time() - START_TIME > SESSION_CAP_SEC:
                save_ckpt()
                STOPPED_EARLY = True
                print("Session time cap reached. Checkpoint saved. Attach this notebook's output as an input and run again to resume.")
                break
        if STOPPED_EARLY:
            break

        # end of epoch: validate on dev
        state["epoch"] = epoch + 1
        state["pos"] = 0
        state["perm"] = None
        entry = {"epoch": epoch, "step": state["step"]}
        if not TRAIN_ON_DEV and "dev" in B["splits"]:
            t_eval = time.time()
            p, r, f1 = quick_dev_f1()
            entry["dev_f1"] = f1
            entry["dev_precision"] = p
            entry["dev_recall"] = r
            print("epoch %d finished | dev Micro-F1@%d = %.4f (P %.4f, R %.4f) | eval %.0fs" % (epoch, TOP_K, f1, p, r, time.time() - t_eval))
            if f1 > BEST_F1:
                BEST_F1 = f1
                BEST_EPOCH = epoch
                BEST_STATE = current_state_dict()
                torch.save(BEST_STATE, BEST_PATH)
                print("  new best on dev")
        else:
            print("epoch", epoch, "finished")
        HISTORY.append(entry)
        save_ckpt()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    TRAIN_SECONDS = state["seconds"]
    if not STOPPED_EARLY:
        print("training done | %.1f min | best dev F1 %.4f at epoch %d" % (TRAIN_SECONDS / 60.0, BEST_F1, BEST_EPOCH))
        if BEST_STATE is None or TRAIN_ON_DEV:
            BEST_STATE = current_state_dict()
            torch.save(BEST_STATE, BEST_PATH)
else:
    if os.path.exists(BEST_PATH):
        BEST_STATE = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
    else:
        found = find_own_file("adapter_best.pt")
        if found is not None:
            BEST_STATE = torch.load(found, map_location="cpu", weights_only=False)
    if BEST_STATE is not None:
        print("loaded saved adapter weights")
    else:
        print("No training and no saved adapter weights: the model is the plain Phase-1 model.")

if not STOPPED_EARLY and BEST_STATE is not None:
    load_state_dict_all(BEST_STATE)
    print("best adapter loaded (dev F1 %.4f at epoch %d)" % (BEST_F1, BEST_EPOCH))
set_eval_mode()

## 10. Dev: sweep the inference options, choose the best, analyse errors

In [ ]:
CHOSEN = {"center": False, "norm": "none", "param": 0, "lambda": 0.0}
SWEEP_ROWS = []
DEV_REPORT = {}

if STOPPED_EARLY:
    print("Training stopped at the session cap: attach this output as an input and re-run to continue. Skipping evaluation.")
elif "dev" in B["splits"]:
    t0 = time.time()
    qids, pids, Qe, Pe = embed_split("dev")
    dev_gold = GOLD["dev"]
    dev_bm25 = bm25_scores("dev", qids, pids)
    SWEEP_ROWS = run_sweep(Qe, Pe, qids, pids, dev_gold, dev_bm25)
    CHOSEN = {"center": SWEEP_ROWS[0]["center"], "norm": SWEEP_ROWS[0]["norm"],
              "param": SWEEP_ROWS[0]["param"], "lambda": SWEEP_ROWS[0]["lambda"]}

    p_plain, r_plain, f1_plain = plain_f1(Qe, Pe, qids, pids, dev_gold)
    print("dev Micro-F1@%d, plain cosine (neural only): %.4f (P %.4f, R %.4f)" % (TOP_K, f1_plain, p_plain, r_plain))
    print("top 8 inference options on dev:")
    for row in SWEEP_ROWS[:8]:
        print("  center=%-5s norm=%-4s param=%-5s bm25_weight=%-4s -> F1 %.4f (P %.4f, R %.4f)" % (
            row["center"], row["norm"], row["param"], row["lambda"], row["f1"], row["precision"], row["recall"]))
    print("chosen:", CHOSEN, "-> dev F1 %.4f" % SWEEP_ROWS[0]["f1"])
    neural_only = []
    for row in SWEEP_ROWS:
        if row["lambda"] == 0.0:
            neural_only.append(row)
    print("best neural-only option: center=%s norm=%s param=%s -> F1 %.4f" % (
        neural_only[0]["center"], neural_only[0]["norm"], neural_only[0]["param"], neural_only[0]["f1"]))
    print("note: the options were picked on dev (%d queries), so dev F1 is slightly optimistic." % len(qids))

    with open(os.path.join(OUT_DIR, "dev_sweep.json"), "w") as f:
        json.dump(SWEEP_ROWS, f)

    # error analysis with the chosen options
    S_final = final_scores(Qe, Pe, qids, pids, CHOSEN, dev_bm25)
    lists = top_ids(S_final, pids, TOP_K)
    dev_lists_100 = top_ids(S_final, pids, 100)
    DEV_REPORT["recall_at"] = {}
    for k in [7, 20, 50, 100]:
        DEV_REPORT["recall_at"][str(k)] = recall_at(S_final, qids, pids, dev_gold, k)
    print("dev recall@k (chosen options):", DEV_REPORT["recall_at"])

    q_len = []
    for q in qids:
        q_len.append(len(B["tokens"]["dev"]["queries"][q]))
    q_len = np.array(q_len)
    cuts = np.percentile(q_len, [33, 66])
    buckets = {"short query": [], "medium query": [], "long query": []}
    gold_buckets = {"1 gold": [], "2-4 gold": [], "5-7 gold": [], ">7 gold": []}
    zero_hit = 0
    for r, q in enumerate(qids):
        g = set(dev_gold[q])
        hits = 0
        for c in lists[r]:
            if c in g:
                hits += 1
        if hits == 0:
            zero_hit += 1
        if q_len[r] <= cuts[0]:
            buckets["short query"].append((hits, len(g)))
        elif q_len[r] <= cuts[1]:
            buckets["medium query"].append((hits, len(g)))
        else:
            buckets["long query"].append((hits, len(g)))
        if len(g) == 1:
            gold_buckets["1 gold"].append((hits, len(g)))
        elif len(g) <= 4:
            gold_buckets["2-4 gold"].append((hits, len(g)))
        elif len(g) <= 7:
            gold_buckets["5-7 gold"].append((hits, len(g)))
        else:
            gold_buckets[">7 gold"].append((hits, len(g)))
    DEV_REPORT["zero_hit_queries"] = zero_hit
    DEV_REPORT["by_query_length"] = {}
    DEV_REPORT["by_gold_count"] = {}
    print("queries with no hit in the top %d: %d / %d" % (TOP_K, zero_hit, len(qids)))
    for title, table, key in [("query length", buckets, "by_query_length"), ("gold count", gold_buckets, "by_gold_count")]:
        for name in table:
            rows = table[name]
            if len(rows) == 0:
                continue
            tp = sum(h for h, g in rows)
            n_gold = sum(g for h, g in rows)
            prec = tp / (TOP_K * len(rows))
            rec = tp / max(n_gold, 1)
            DEV_REPORT[key][name] = {"queries": len(rows), "precision": prec, "recall": rec}
            print("  %-13s %-13s n=%3d  P@%d %.3f  R@%d %.3f" % (title, name, len(rows), TOP_K, prec, TOP_K, rec))

    with open(os.path.join(OUT_DIR, "dev_predictions.json"), "w") as f:
        out = {}
        for r, q in enumerate(qids):
            out[q] = lists[r]
        json.dump(out, f)
    print("dev analysis took %.0fs" % (time.time() - t0))
else:
    print("No dev split: using plain cosine for the test predictions.")

## 11. Test predictions
Exactly 7 unique ids per test query, taken only from the test pool, ordered best first. The query itself is never returned.

In [ ]:
TEST_SECONDS = 0.0
PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
if STOPPED_EARLY:
    print("Skipped (training not finished).")
elif RUN_TEST and "test" in B["splits"]:
    t0 = time.time()
    qids, pids, Qe, Pe = embed_split("test")
    test_bm25 = bm25_scores("test", qids, pids)
    TEST_SECONDS = time.time() - t0

    S_final = final_scores(Qe, Pe, qids, pids, CHOSEN, test_bm25)
    final_lists = top_ids(S_final, pids, TOP_K)
    S_plain = make_scores(Qe, Pe, qids, pids, False)
    plain_lists = top_ids(S_plain, pids, TOP_K)

    def build_prediction(lists):
        pred = {}
        pool_set = set(pids)
        for r, q in enumerate(qids):
            ranked = lists[r]
            if len(ranked) != TOP_K or len(set(ranked)) != TOP_K:
                raise RuntimeError("bad ranking for query " + q)
            for c in ranked:
                if c not in pool_set:
                    raise RuntimeError("candidate outside the test pool for query " + q)
                if c == q:
                    raise RuntimeError("query retrieved itself: " + q)
            pred[q] = ranked
        if len(pred) != len(B["query_ids"]["test"]):
            raise RuntimeError("missing test queries")
        return pred

    prediction = build_prediction(final_lists)
    with open(PRED_PATH, "w") as f:
        json.dump(prediction, f)
    with open(os.path.join(OUT_DIR, "prediction_plain_cosine.json"), "w") as f:
        json.dump(build_prediction(plain_lists), f)
    zip_path = os.path.join(OUT_DIR, "submission.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(PRED_PATH, "prediction.json")
    print("wrote prediction.json for %d test queries (chosen options: %s)" % (len(prediction), CHOSEN))
    print("wrote prediction_plain_cosine.json (neural cosine only) and submission.zip")
    print("test embedding + scoring: %.0fs for %d queries and %d pool docs" % (TEST_SECONDS, len(qids), len(pids)))
else:
    print("RUN_TEST is off or there is no test split.")

## 12. Run summary (for the report)

In [ ]:
summary = {
    "model": "Phase-1 Model 1 (GQA)",
    "adapter": "LoRA",
    "kd": True,
    "teachers": TEACHER_KEYS,
    "teacher_dev_f1": TEACH_DEV_F1,
    "teacher_info": {k: RAW["_info"][k] for k in TEACHER_KEYS if k in RAW["_info"]},
    "kd_weight": KD_WEIGHT,
    "kd_temperature": KD_TEMPERATURE,
    "train_on_dev": TRAIN_ON_DEV,
    "adapter_settings": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "base_params": int(N_BASE),
    "adapter_params": int(N_ADAPTER),
    "projection_params": int(D_MODEL * D_MODEL) if PROJ_MODULE is not None else 0,
    "embedding_dim": int(D_MODEL),
    "doc_view": DOC_VIEW,
    "head_chunks": HEAD_CHUNKS,
    "tail_chunks": TAIL_CHUNKS,
    "chunk_len": CHUNK_LEN,
    "pooling": POOLING,
    "temperature": TEMPERATURE,
    "queries_per_step": QUERIES_PER_STEP,
    "pos_per_query": POS_PER_QUERY,
    "hard_neg_per_query": HARD_NEG_PER_QUERY,
    "hard_neg_range": list(HARD_NEG_RANGE),
    "epochs": EPOCHS,
    "lr": LR,
    "train_seconds": TRAIN_SECONDS,
    "stopped_early": STOPPED_EARLY,
    "history": HISTORY,
    "best_dev_f1_plain_cosine": BEST_F1,
    "best_epoch": BEST_EPOCH,
    "zero_shot_dev": ZERO_SHOT,
    "chosen_inference_options": CHOSEN,
    "dev_best_sweep_row": SWEEP_ROWS[0] if len(SWEEP_ROWS) > 0 else None,
    "dev_report": DEV_REPORT,
    "test_embed_seconds": TEST_SECONDS,
    "peak_gpu_mem_gb": (torch.cuda.max_memory_allocated() / 1e9) if DEVICE.type == "cuda" else None,
}
with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str)[:3000])
print("\nfiles in", OUT_DIR)
for name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, name)
    if os.path.isfile(full):
        print("  %-30s %.2f MB" % (name, os.path.getsize(full) / 1e6))